# Fish2 Raphe morphology clustering — v20 Raphe-superior soma-in-ROI only

This notebook re-runs the morphology/NBLAST clustering **only for neurons touching the exact Raphe-superior ROIs whose soma/root is inside the Raphe-superior region**.

Important distinction:
- `NC(rois=..., roi_req='any')` returns neurons whose neurites/synapses touch Raphe. Their somas can be far away.
- This v20 notebook first pulls exact Raphe-superior-touching neurons, then filters to cells whose soma/root is in the Raphe-superior region.
- If Fish2 exposes an explicit soma-ROI column, the notebook uses that. Otherwise it uses a transparent spatial proxy based on in-ROI synapse/contact coordinates, because the Raphe-superior ROI mesh is not available from neuPrint for point-in-mesh filtering.

After clustering, the notebook loads its own outputs into the same plotting/figure-ready machinery as v19: UMAP/MDS, cluster colors, three 3D brain views, figure-ready projection panels, custom cluster export, and static PDFs.


**Update:** brain shell rendering now keeps only the largest connected shell component to remove detached ROI/mask blobs, and uses a lower transparency for a cleaner figure-ready outline.

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
SKELETON_DIR = PROJECT_DIR / "skeletons"
OUT_ROOT = PROJECT_DIR / "fish2_raphe_morphology_v20_rapheSuperior_somaInROI_outputs"

SELECTED_IDS_CSV = DATA_DIR / "raphe_234_ids.csv"
SELECTED_RAW_SWC_DIR = SKELETON_DIR / "selected_raw_swc"
RAPHE_SUPERIOR_SOMA_IN_ROI_RAW_SWC_DIR = SKELETON_DIR / "raphe_superior_soma_in_roi_raw_swc"
NEW_CELLS_RAW_SWC_DIR = SKELETON_DIR / "new_cells_raw_swc"
NEW_CELL_IDS_CSV = DATA_DIR / "new_cell_ids.csv"

for p in [DATA_DIR, SKELETON_DIR, OUT_ROOT, SELECTED_RAW_SWC_DIR, RAPHE_SUPERIOR_SOMA_IN_ROI_RAW_SWC_DIR, NEW_CELLS_RAW_SWC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# This notebook runs only the soma-in-Raphe-superior dataset.
RUN_SELECTED_RAPHE = False
RUN_RAPHE_SUPERIOR_SOMA_IN_ROI = True

# neuPrint / fishFuncEM
NEUPRINT_SERVER = "https://neuprint-fish2.janelia.org"
NEUPRINT_DATASET = "fish2"
NEUPRINT_TOKEN_FALLBACK = ""   # Prefer env var NEUPRINT_TOKEN or local neuprint_token.txt

# Exact Raphe-superior ROIs to use.
# Keep these exact; do not use every ROI containing "Raphe" unless you explicitly want inferior+superior together.
RAPHE_SUPERIOR_EXACT_ROIS = ["Raphe_Superior", "Raphe_Superior1"]
MAX_RAPHE_SUPERIOR_CELLS = None  # set e.g. 500 while testing; None for all

# Soma-in-ROI filtering.
# First the notebook tries explicit soma ROI columns if neuPrint returns them.
# If not available, it uses a transparent spatial proxy from in-ROI synapse/contact coordinates.
SOMA_IN_ROI_FILTER_MODE = "explicit_then_synapse_cloud_proxy"
SOMA_PROXY_PERCENTILE_LOW = 1.0
SOMA_PROXY_PERCENTILE_HIGH = 99.0
SOMA_PROXY_PADDING = 1200.0
SOMA_PROXY_USE_ELLIPSOID = True
SOMA_PROXY_RADIUS_SCALE = 1.25
SOMA_PROXY_MIN_RADIUS = 1500.0
SOMA_PROXY_MAX_RADIUS = 9000.0
SOMA_PROXY_MAX_NEAREST_CONTACT_DISTANCE = None  # set e.g. 3500 to be stricter; None disables this gate
SOMA_PROXY_MAX_CONTACT_POINTS_FOR_FIT = 80000
SOMA_PROXY_RANDOM_SEED = 11

# Download/cache SWCs for filtered soma-in-ROI cells if missing.
FETCH_MISSING_SWCS = True
REFETCH_EXISTING_SWCS = False
FETCH_HEAL = False
FETCH_MAX_RETRIES = 3
FETCH_SLEEP_SEC = 0.2

# QC: tuned to remove obvious stick-like fragments but not over-prune.
QC_MIN_NODES = 120
QC_MIN_CABLE = 750.0
QC_MIN_BRANCHPOINTS = 3
QC_MIN_ENDPOINTS = 4
QC_MAX_ROOTS = 4
QC_MAX_LONGEST_PATH_FRACTION = 0.88
QC_MIN_NODE_DENSITY_QUANTILE = 0.15
QC_KEEP_TOP_FRAC_AFTER_HARD_GATE = 0.75
QC_MIN_KEEP = 20
QC_MAX_KEEP = None

# fishFuncEM NBLAST
FISHFUNCEM_MIRROR = True
FORCE_RECOMPUTE_NBLAST = False
DISTANCE_MODE = "rank"

# Clustering: coarse automatic threshold, not fixed number of clusters.
AUTO_MIN_REAL_CLUSTERS = 3
AUTO_MAX_REAL_CLUSTERS = 12
AUTO_MAX_SINGLETON_FRAC = 0.35
AUTO_MIN_MULTI_CLUSTER_SIZE = 2
AUTO_ALLOW_LARGE_CLUSTERS = True
RUN_MAXCLUST_SWEEP = False

# Embeddings / figures
RANDOM_SEED = 42
FIG_DPI = 300
SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = True
UMAP_N_NEIGHBORS = 12
UMAP_MIN_DIST = 0.12

# v14 rendering disabled because the v19-style figure-ready plotting section is appended at the end.
MAKE_CLUSTER_RENDERINGS = False
MAX_CLUSTERS_TO_RENDER = None
MAX_CELLS_PER_CLUSTER_RENDER = 40
SHOW_SOMA_MARKERS = True
SHOW_BRAIN_SHELL = True
SHELL_ALPHA = 0.055
CELL_LINE_WIDTH = 3
POPULATION_CONTEXT_ALPHA = 0.04

# Optional static 3D export requires kaleido.
SAVE_INTERACTIVE_HTML = True
SAVE_STATIC_3D = False

print("Output folder:", OUT_ROOT)
print("Exact Raphe-superior ROIs:", RAPHE_SUPERIOR_EXACT_ROIS)


In [ ]:
# ============================================================
# 2. Imports and environment check
# ============================================================

import os, re, time, json, math, warnings, traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from tqdm import tqdm

from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.sparse import coo_matrix
from scipy.ndimage import binary_fill_holes
from skimage.measure import marching_cubes

from sklearn.manifold import MDS
try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

import plotly.graph_objects as go

print("Python:", os.sys.version)

try:
    from neuprint import Client, fetch_neurons, NeuronCriteria as NC
    print("neuprint-python: OK")
except Exception as e:
    print("neuprint-python import failed:", repr(e))

try:
    from fishfuncem import NeuprintServer, FishNeuronMorph
    print("fishfuncem: OK")
except Exception as e:
    raise ImportError("fishfuncem is required for this notebook. Use the Python >=3.11 environment.") from e

try:
    import navis
    HAS_NAVIS = True
    print("navis: OK")
except Exception as e:
    HAS_NAVIS = False
    print("navis unavailable; SWC parsing/rendering still works with built-in parser:", repr(e))

try:
    import openpyxl
    HAS_OPENPYXL = True
except Exception:
    HAS_OPENPYXL = False

try:
    from neuprint import fetch_synapses, SynapseCriteria as SC
    HAS_FETCH_SYNAPSES = True
    print("neuprint fetch_synapses: OK")
except Exception as e:
    HAS_FETCH_SYNAPSES = False
    print("neuprint fetch_synapses unavailable:", repr(e))

try:
    from scipy.spatial import cKDTree
    HAS_CKDTREE = True
except Exception:
    HAS_CKDTREE = False


In [ ]:
# ============================================================
# 3. Token + neuPrint client helpers
# ============================================================

def get_neuprint_token() -> str | None:
    """
    Token order:
    1) NEUPRINT_TOKEN environment variable
    2) neuprint_token.txt in the notebook/project directory
    3) NEUPRINT_TOKEN_FALLBACK in the config cell

    IMPORTANT:
    fishFuncEM worked in the v9.9 notebook when the raw token string was placed
    directly in BOTH NEUPRINT_TOKEN and NEUPRINT_APPLICATION_CREDENTIALS.
    Do not replace NEUPRINT_APPLICATION_CREDENTIALS with a generated JSON file here.
    """
    tok = os.environ.get("NEUPRINT_TOKEN", "").strip()
    if tok:
        return tok

    token_file = PROJECT_DIR / "neuprint_token.txt"
    if token_file.exists():
        tok = token_file.read_text().strip()
        if tok:
            return tok

    if str(NEUPRINT_TOKEN_FALLBACK).strip():
        return str(NEUPRINT_TOKEN_FALLBACK).strip()

    return None


def configure_token_env_for_neuprint_and_fishfuncem():
    """
    Keep this deliberately simple, matching the working selected-only v9.9 notebook.

    fishFuncEM's NeuprintServer() creates its own neuPrint client, so we make sure
    both environment variables contain the raw token before constructing NeuprintServer().
    """
    token = get_neuprint_token()
    if not token:
        raise RuntimeError(
            "No neuPrint token found. Set NEUPRINT_TOKEN, create neuprint_token.txt, "
            "or paste token into NEUPRINT_TOKEN_FALLBACK."
        )

    os.environ["NEUPRINT_TOKEN"] = token
    os.environ["NEUPRINT_APPLICATION_CREDENTIALS"] = token

    print("Token found and exported for neuprint-python + fishFuncEM.")
    print("  NEUPRINT_TOKEN present:", bool(os.environ.get("NEUPRINT_TOKEN")))
    print("  NEUPRINT_APPLICATION_CREDENTIALS present:", bool(os.environ.get("NEUPRINT_APPLICATION_CREDENTIALS")))

    return token


def get_neuprint_client():
    token = configure_token_env_for_neuprint_and_fishfuncem()
    return Client(NEUPRINT_SERVER, dataset=NEUPRINT_DATASET, token=token)


c = get_neuprint_client()
print("Connected to:", NEUPRINT_SERVER, NEUPRINT_DATASET)


In [ ]:
# ============================================================
# 4. SWC parser, soma estimate, QC metrics
# ============================================================

SWC_COLS = ["node_id", "type", "x", "y", "z", "radius", "parent"]


def body_id_from_path(path):
    m = re.search(r"(\d+)", Path(path).stem)
    return int(m.group(1)) if m else None


def read_swc(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 7:
                continue
            try:
                rows.append([int(float(parts[0])), int(float(parts[1])), float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5]), int(float(parts[6]))])
            except Exception:
                pass
    df = pd.DataFrame(rows, columns=SWC_COLS)
    return df


def swc_edges(df):
    if df is None or df.empty:
        return []
    ids = set(df["node_id"].astype(int))
    edges = []
    for _, r in df.iterrows():
        p = int(r["parent"])
        n = int(r["node_id"])
        if p in ids and p != -1:
            edges.append((p, n))
    return edges


def swc_cable_length(df):
    if df is None or df.empty:
        return 0.0
    coords = df.set_index("node_id")[["x", "y", "z"]]
    total = 0.0
    for p, n in swc_edges(df):
        try:
            total += float(np.linalg.norm(coords.loc[n].values - coords.loc[p].values))
        except Exception:
            pass
    return total


def swc_branch_endpoint_root_counts(df):
    if df is None or df.empty:
        return 0, 0, 0
    ids = set(df["node_id"].astype(int))
    parents = df["parent"].astype(int).values
    node_ids = df["node_id"].astype(int).values
    child_count = defaultdict(int)
    roots = 0
    for n, p in zip(node_ids, parents):
        if p == -1 or p not in ids:
            roots += 1
        else:
            child_count[p] += 1
    n_branch = sum(v >= 2 for v in child_count.values())
    n_end = sum(child_count.get(n, 0) == 0 for n in node_ids)
    return int(n_branch), int(n_end), int(roots)


def longest_path_fraction_approx(df):
    # Approximate with tree diameter on weighted SWC graph. Safe enough for QC.
    edges = swc_edges(df)
    cable = swc_cable_length(df)
    if not edges or cable <= 0:
        return np.nan
    coords = df.set_index("node_id")[["x", "y", "z"]]
    adj = defaultdict(list)
    for a, b in edges:
        w = float(np.linalg.norm(coords.loc[a].values - coords.loc[b].values))
        adj[a].append((b, w)); adj[b].append((a, w))
    def farthest(start):
        seen = {start: 0.0}; stack = [start]
        while stack:
            u = stack.pop()
            for v, w in adj[u]:
                if v not in seen:
                    seen[v] = seen[u] + w
                    stack.append(v)
        if not seen:
            return start, 0.0
        k = max(seen, key=seen.get)
        return k, seen[k]
    start = next(iter(adj))
    a, _ = farthest(start)
    b, diam = farthest(a)
    return float(diam / cable) if cable else np.nan


def estimate_soma_from_swc(df):
    if df is None or df.empty:
        return None
    # Prefer SWC soma type 1.
    soma = df[df["type"].astype(int) == 1]
    if len(soma):
        r = soma.iloc[0]
        return np.array([r.x, r.y, r.z], dtype=float)
    # Then root.
    roots = df[df["parent"].astype(int) == -1]
    if len(roots):
        r = roots.iloc[0]
        return np.array([r.x, r.y, r.z], dtype=float)
    # Then largest radius.
    if "radius" in df.columns and df["radius"].notna().any():
        r = df.loc[df["radius"].idxmax()]
        return np.array([r.x, r.y, r.z], dtype=float)
    return df[["x", "y", "z"]].median().values.astype(float)


def compute_swc_metrics(path):
    bid = body_id_from_path(path)
    try:
        df = read_swc(path)
        n_nodes = len(df)
        cable = swc_cable_length(df)
        n_branch, n_end, n_roots = swc_branch_endpoint_root_counts(df)
        lpf = longest_path_fraction_approx(df)
        density = n_nodes / cable if cable > 0 else np.nan
        soma = estimate_soma_from_swc(df)
        return {
            "bodyId": bid,
            "swc_path": str(path),
            "n_nodes": n_nodes,
            "cable_length": cable,
            "n_branchpoints": n_branch,
            "n_endpoints": n_end,
            "n_roots": n_roots,
            "node_density": density,
            "longest_path_fraction": lpf,
            "soma_x": np.nan if soma is None else soma[0],
            "soma_y": np.nan if soma is None else soma[1],
            "soma_z": np.nan if soma is None else soma[2],
            "read_ok": True,
            "error": "",
        }
    except Exception as e:
        return {"bodyId": bid, "swc_path": str(path), "read_ok": False, "error": repr(e)}


def build_qc_table(swc_dir, label):
    paths = sorted(Path(swc_dir).glob("*.swc"))
    rows = [compute_swc_metrics(p) for p in tqdm(paths, desc=f"QC {label}")]
    qc = pd.DataFrame(rows)
    if qc.empty:
        return qc
    for col in ["n_nodes", "cable_length", "n_branchpoints", "n_endpoints", "n_roots", "node_density", "longest_path_fraction"]:
        if col in qc.columns:
            qc[col] = pd.to_numeric(qc[col], errors="coerce")
    hard = (
        (qc.get("read_ok", False) == True) &
        (qc["n_nodes"] >= QC_MIN_NODES) &
        (qc["cable_length"] >= QC_MIN_CABLE) &
        (qc["n_branchpoints"] >= QC_MIN_BRANCHPOINTS) &
        (qc["n_endpoints"] >= QC_MIN_ENDPOINTS) &
        (qc["n_roots"] <= QC_MAX_ROOTS) &
        (qc["longest_path_fraction"].fillna(1) <= QC_MAX_LONGEST_PATH_FRACTION)
    )
    qc["qc_pass_hard"] = hard
    cand = qc[hard].copy()
    if len(cand):
        density_cut = cand["node_density"].quantile(QC_MIN_NODE_DENSITY_QUANTILE)
        cand = cand[cand["node_density"] >= density_cut].copy()
        cand["rank_density"] = cand["node_density"].rank(pct=True)
        cand["rank_nodes"] = cand["n_nodes"].rank(pct=True)
        cand["rank_branch"] = cand["n_branchpoints"].rank(pct=True)
        cand["rank_not_stringy"] = (1 - cand["longest_path_fraction"].rank(pct=True))
        cand["qc_quality_score"] = 0.35*cand["rank_density"] + 0.25*cand["rank_nodes"] + 0.20*cand["rank_branch"] + 0.20*cand["rank_not_stringy"]
        n_keep = max(QC_MIN_KEEP, int(round(len(cand) * QC_KEEP_TOP_FRAC_AFTER_HARD_GATE)))
        n_keep = min(n_keep, len(cand))
        if QC_MAX_KEEP is not None:
            n_keep = min(n_keep, QC_MAX_KEEP)
        keep_ids = set(cand.sort_values("qc_quality_score", ascending=False).head(n_keep)["bodyId"].astype(int))
    else:
        keep_ids = set()
    qc["qc_pass"] = qc["bodyId"].astype("Int64").isin(keep_ids)
    qc["qc_reason"] = np.where(qc["qc_pass"], "pass", "fail_qc_or_density")
    out = OUT_ROOT / f"{label}_swc_qc.csv"
    qc.to_csv(out, index=False)
    print(f"{label}: QC pass {int(qc['qc_pass'].sum())} / {len(qc)} | saved {out}")
    return qc

In [ ]:
# ============================================================
# 5. Dataset loading, soma-in-Raphe-superior filtering, and SWC fetching
# ============================================================


def load_selected_ids():
    if not SELECTED_IDS_CSV.exists():
        ids = [body_id_from_path(p) for p in SELECTED_RAW_SWC_DIR.glob("*.swc")]
        ids = sorted([int(x) for x in ids if x is not None])
        print(f"No {SELECTED_IDS_CSV}; using {len(ids)} IDs from selected SWC filenames.")
        return ids
    df = pd.read_csv(SELECTED_IDS_CSV)
    if "bodyId" in df.columns:
        ids = df["bodyId"].dropna().astype(int).tolist()
    else:
        ids = df.iloc[:, 0].dropna().astype(int).tolist()
    ids = list(dict.fromkeys(ids))
    print(f"Loaded selected IDs: {len(ids)}")
    return ids


def valid_exact_raphe_superior_rois(client):
    roi_names = set(client.meta["roiInfo"].keys())
    valid = [r for r in RAPHE_SUPERIOR_EXACT_ROIS if r in roi_names]
    missing = [r for r in RAPHE_SUPERIOR_EXACT_ROIS if r not in roi_names]
    if missing:
        print("Requested ROI names missing from neuPrint metadata:", missing)
    if not valid:
        raphe_like = sorted([r for r in roi_names if "raphe" in r.lower()])
        print("Available Raphe-like ROIs:")
        for r in raphe_like:
            print("  ", r)
        raise RuntimeError("No valid exact Raphe-superior ROI names found.")
    print("Using exact Raphe-superior ROIs:")
    for r in valid:
        print("  ", r)
    return valid


def fetch_body_ids_touching_rois(client, rois, label):
    neurons, roi_info = fetch_neurons(NC(rois=rois, roi_req="any"), client=client)
    neurons = neurons.copy()
    neurons["bodyId"] = neurons["bodyId"].astype(int)
    neurons = neurons.drop_duplicates("bodyId").sort_values("bodyId")
    if MAX_RAPHE_SUPERIOR_CELLS is not None:
        neurons = neurons.iloc[:MAX_RAPHE_SUPERIOR_CELLS].copy()
    out = OUT_ROOT / f"{label}_all_touching_bodyIds_from_neuprint.csv"
    neurons.to_csv(out, index=False)
    print(f"{label}: fetched {len(neurons)} body IDs touching {rois}; saved {out}")
    return neurons, roi_info


def location_to_xyz(value):
    """Robustly convert neuPrint point-like values to np.array([x,y,z])."""
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, dict):
        if {"x", "y", "z"}.issubset(value):
            return np.array([value["x"], value["y"], value["z"]], dtype=float)
        if "coordinates" in value and len(value["coordinates"]) >= 3:
            return np.array(value["coordinates"][:3], dtype=float)
    for attrs in [("x", "y", "z"), ("X", "Y", "Z")]:
        if all(hasattr(value, a) for a in attrs):
            return np.array([getattr(value, attrs[0]), getattr(value, attrs[1]), getattr(value, attrs[2])], dtype=float)
    if isinstance(value, (list, tuple, np.ndarray)) and len(value) >= 3:
        return np.array(value[:3], dtype=float)
    # String fallback: Point{z:..., y:..., x:...}, [x,y,z], etc.
    s = str(value)
    nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", s)
    if len(nums) >= 3:
        vals = np.array([float(n) for n in nums[:3]], dtype=float)
        return vals
    return None


def add_soma_xyz_columns(neurons):
    neurons = neurons.copy()
    loc_col = None
    for col in ["somaLocation", "tosomaLocation", "rootLocation", "location"]:
        if col in neurons.columns:
            loc_col = col
            break
    if loc_col is None:
        print("No soma/root location column found in neuron table.")
        neurons["soma_x"] = np.nan; neurons["soma_y"] = np.nan; neurons["soma_z"] = np.nan
        return neurons, None
    xyz = neurons[loc_col].apply(location_to_xyz)
    neurons["soma_x"] = [float(v[0]) if v is not None and np.isfinite(v).all() else np.nan for v in xyz]
    neurons["soma_y"] = [float(v[1]) if v is not None and np.isfinite(v).all() else np.nan for v in xyz]
    neurons["soma_z"] = [float(v[2]) if v is not None and np.isfinite(v).all() else np.nan for v in xyz]
    print(f"Using {loc_col} as soma/root coordinate column. Coordinates available for {neurons[['soma_x','soma_y','soma_z']].notna().all(axis=1).sum()} / {len(neurons)} cells.")
    return neurons, loc_col


def explicit_soma_roi_filter(neurons, rois):
    """Use explicit soma ROI columns if Fish2 neuPrint exposes them."""
    roi_cols = [c for c in neurons.columns if "soma" in c.lower() and "roi" in c.lower()]
    if not roi_cols:
        return None, None
    roi_set = set(rois)
    for col in roi_cols:
        vals = neurons[col]
        keep = vals.apply(lambda x: any(r in str(x) for r in roi_set) if pd.notna(x) else False)
        if keep.any():
            out = neurons.loc[keep].copy()
            out["soma_in_roi_filter_source"] = f"explicit column {col}"
            print(f"Explicit soma ROI filter using column {col}: kept {len(out)} / {len(neurons)}")
            return out, col
    print("Soma ROI-like columns existed but none matched requested ROIs:", roi_cols)
    return None, roi_cols


def make_synapse_criteria_for_rois(rois):
    """
    Build SynapseCriteria for these ROIs, allowing non-primary ROIs.

    In Fish2, Raphe_Superior / Raphe_Superior1 are non-primary ROIs in neuPrint.
    Some neuprint-python versions use `primary_only=False`; some mention
    `include_nonprimary=True` in the error message. This helper handles both.
    """
    try:
        return SC(rois=rois, primary_only=False)
    except TypeError:
        # For newer/alternate neuprint-python APIs, if present.
        return SC(rois=rois, include_nonprimary=True)


def synapse_xyz_table_for_rois(body_ids, rois, batch_size=400):
    """
    Fetch synapse/contact coordinates inside the requested Raphe-superior ROIs.

    Important: Raphe_Superior and Raphe_Superior1 are non-primary ROIs in Fish2,
    so SynapseCriteria must be created with primary_only=False. The old version
    used SC(rois=rois), which raises:
        AssertionError: You listed non-primary ROIs ...
    """
    if not HAS_FETCH_SYNAPSES:
        raise RuntimeError("fetch_synapses / SynapseCriteria not available, cannot build synapse-cloud soma proxy.")

    body_ids = [int(x) for x in body_ids]
    print(f"Fetching in-ROI synapse/contact points for {len(body_ids):,} bodies in {rois}...")
    print("Using SynapseCriteria(..., primary_only=False) because these Raphe ROIs are non-primary.")

    syn_batches = []
    sc = make_synapse_criteria_for_rois(rois)

    for start in range(0, len(body_ids), batch_size):
        batch = body_ids[start:start + batch_size]
        print(f"  batch {start//batch_size + 1:03d}: bodies {start + 1:,}-{start + len(batch):,} / {len(body_ids):,}")
        try:
            syn_b = fetch_synapses(NC(bodyId=batch), sc, client=c)
        except Exception as e:
            print(f"    batch failed: {repr(e)}")
            continue
        if syn_b is not None and len(syn_b):
            syn_batches.append(syn_b.copy())

    if not syn_batches:
        raise RuntimeError("No in-ROI synapse/contact points were returned for the requested bodies/ROIs.")

    syn = pd.concat(syn_batches, ignore_index=True)

    if {"x", "y", "z"}.issubset(syn.columns):
        pass
    elif "location" in syn.columns:
        xyz = syn["location"].apply(location_to_xyz)
        syn["x"] = [v[0] if v is not None else np.nan for v in xyz]
        syn["y"] = [v[1] if v is not None else np.nan for v in xyz]
        syn["z"] = [v[2] if v is not None else np.nan for v in xyz]
    else:
        print("Synapse table columns:", list(syn.columns))
        raise RuntimeError("Could not find x/y/z or location columns in synapse table.")

    syn = syn.dropna(subset=["x", "y", "z"])
    for col in ["x", "y", "z"]:
        syn[col] = syn[col].astype(float)

    out = OUT_ROOT / "raphe_superior_in_roi_synapse_contact_points.csv"
    syn.to_csv(out, index=False)
    print(f"In-ROI synapse/contact points: {len(syn):,}; saved {out}")
    return syn


def filter_somas_by_synapse_cloud_proxy(neurons, rois):
    """
    Approximate soma-in-Raphe-superior when no ROI mesh or explicit soma ROI exists.
    Builds a robust spatial cloud from synapse/contact points inside exact Raphe-superior ROIs
    and keeps cells whose soma/root coordinates fall within that cloud.
    """
    neurons, loc_col = add_soma_xyz_columns(neurons)
    coords = neurons[["soma_x", "soma_y", "soma_z"]].to_numpy(float)
    valid_soma = np.isfinite(coords).all(axis=1)
    if valid_soma.sum() == 0:
        raise RuntimeError("No soma/root coordinates available, cannot filter by soma-in-ROI proxy.")

    body_ids = neurons["bodyId"].astype(int).tolist()
    syn = synapse_xyz_table_for_rois(body_ids, rois)
    pts = syn[["x", "y", "z"]].to_numpy(float)
    pts = pts[np.isfinite(pts).all(axis=1)]
    if len(pts) == 0:
        raise RuntimeError("No in-ROI synapse/contact coordinates found.")
    if len(pts) > SOMA_PROXY_MAX_CONTACT_POINTS_FOR_FIT:
        rng = np.random.default_rng(SOMA_PROXY_RANDOM_SEED)
        pts_fit = pts[rng.choice(len(pts), SOMA_PROXY_MAX_CONTACT_POINTS_FOR_FIT, replace=False)]
    else:
        pts_fit = pts

    lo = np.percentile(pts_fit, SOMA_PROXY_PERCENTILE_LOW, axis=0) - SOMA_PROXY_PADDING
    hi = np.percentile(pts_fit, SOMA_PROXY_PERCENTILE_HIGH, axis=0) + SOMA_PROXY_PADDING
    in_bbox = valid_soma & np.all((coords >= lo[None, :]) & (coords <= hi[None, :]), axis=1)

    center = np.median(pts_fit, axis=0)
    keep = in_bbox.copy()
    radii = None
    eigvecs = None
    if SOMA_PROXY_USE_ELLIPSOID:
        cov = np.cov((pts_fit - center[None, :]).T)
        eigvals, eigvecs = np.linalg.eigh(cov)
        order = np.argsort(eigvals)[::-1]
        eigvals = eigvals[order]
        eigvecs = eigvecs[:, order]
        radii = np.sqrt(np.maximum(eigvals, 1e-9)) * SOMA_PROXY_RADIUS_SCALE
        radii = np.clip(radii, SOMA_PROXY_MIN_RADIUS, SOMA_PROXY_MAX_RADIUS)
        centered = coords - center[None, :]
        uvw = centered @ eigvecs
        ell = np.sum((uvw / radii[None, :]) ** 2, axis=1) <= 1.0
        keep = valid_soma & ell
        # Union with very conservative bbox can keep some edge cases if ellipsoid is too tight.
        keep = keep | in_bbox

    nearest_dist = np.full(len(neurons), np.nan)
    if SOMA_PROXY_MAX_NEAREST_CONTACT_DISTANCE is not None and HAS_CKDTREE:
        tree = cKDTree(pts_fit)
        d, _ = tree.query(coords[valid_soma], k=1)
        nearest_dist[valid_soma] = d
        keep = keep & (nearest_dist <= float(SOMA_PROXY_MAX_NEAREST_CONTACT_DISTANCE))

    out = neurons.loc[keep].copy()
    out["soma_in_roi_filter_source"] = "synapse/contact cloud spatial proxy"
    out["soma_proxy_nearest_contact_distance"] = nearest_dist[keep]
    out["soma_proxy_in_bbox"] = in_bbox[keep]

    proxy = {
        "bbox_low": lo,
        "bbox_high": hi,
        "center": center,
        "radii": radii if radii is not None else np.array([np.nan, np.nan, np.nan]),
        "eigvecs": eigvecs if eigvecs is not None else np.eye(3),
    }
    np.savez_compressed(OUT_ROOT / "raphe_superior_soma_proxy_geometry.npz", **proxy)

    print("Soma-in-Raphe-superior proxy filter summary:")
    print(f"  touching exact superior ROIs: {len(neurons):,}")
    print(f"  valid soma/root coords: {valid_soma.sum():,}")
    print(f"  kept soma-in-region proxy: {len(out):,}")
    print(f"  bbox low: {lo}")
    print(f"  bbox high: {hi}")
    print(f"  ellipsoid center: {center}")
    print(f"  ellipsoid radii: {proxy['radii']}")
    return out, syn


def filter_neurons_to_soma_in_raphe_superior(neurons, rois):
    neurons = neurons.copy()
    neurons, loc_col = add_soma_xyz_columns(neurons)

    if SOMA_IN_ROI_FILTER_MODE in ["explicit", "explicit_then_synapse_cloud_proxy"]:
        explicit, used_col = explicit_soma_roi_filter(neurons, rois)
        if explicit is not None and len(explicit):
            return explicit, None
        if SOMA_IN_ROI_FILTER_MODE == "explicit":
            raise RuntimeError("Explicit soma ROI filter requested but no usable soma ROI column was found.")

    filtered, syn = filter_somas_by_synapse_cloud_proxy(neurons, rois)
    return filtered, syn


def fetch_swc_for_ids(client, body_ids, out_dir, label):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    logs = []
    for bid in tqdm([int(x) for x in body_ids], desc=f"Fetch SWC {label}"):
        out_path = out_dir / f"{bid}.swc"
        if out_path.exists() and not REFETCH_EXISTING_SWCS:
            logs.append({"bodyId": bid, "status": "exists", "swc_path": str(out_path)})
            continue
        ok = False; err = ""
        if FETCH_MISSING_SWCS:
            for attempt in range(1, FETCH_MAX_RETRIES+1):
                try:
                    client.fetch_skeleton(bid, heal=FETCH_HEAL, export_path=str(out_path))
                    ok = True; break
                except Exception as e:
                    err = repr(e); time.sleep(FETCH_SLEEP_SEC)
        logs.append({"bodyId": bid, "status": "fetched" if ok else "failed", "swc_path": str(out_path), "error": err})
    log = pd.DataFrame(logs)
    log.to_csv(OUT_ROOT / f"{label}_swc_fetch_log.csv", index=False)
    return log


selected_ids = load_selected_ids() if RUN_SELECTED_RAPHE else []

if RUN_RAPHE_SUPERIOR_SOMA_IN_ROI:
    raphe_superior_rois = valid_exact_raphe_superior_rois(c)
    raphe_superior_all_touching_neurons, raphe_superior_roi_info = fetch_body_ids_touching_rois(
        c, raphe_superior_rois, "raphe_superior_exact"
    )
    raphe_superior_soma_neurons, raphe_superior_in_roi_contacts = filter_neurons_to_soma_in_raphe_superior(
        raphe_superior_all_touching_neurons, raphe_superior_rois
    )
    raphe_superior_soma_neurons = raphe_superior_soma_neurons.drop_duplicates("bodyId").sort_values("bodyId")
    soma_out = OUT_ROOT / "raphe_superior_soma_in_roi_bodyIds_from_neuprint.csv"
    raphe_superior_soma_neurons.to_csv(soma_out, index=False)
    raphe_superior_soma_in_roi_ids = raphe_superior_soma_neurons["bodyId"].astype(int).tolist()
    print(f"Saved soma-in-Raphe-superior IDs: {soma_out}")
    fetch_swc_for_ids(c, raphe_superior_soma_in_roi_ids, RAPHE_SUPERIOR_SOMA_IN_ROI_RAW_SWC_DIR, "raphe_superior_soma_in_roi")
else:
    raphe_superior_rois = []
    raphe_superior_soma_neurons = pd.DataFrame()
    raphe_superior_soma_in_roi_ids = []

print("Selected IDs:", len(selected_ids))
print("Raphe superior soma-in-ROI IDs:", len(raphe_superior_soma_in_roi_ids))


### Fix note: non-primary Raphe ROIs

`Raphe_Superior` and `Raphe_Superior1` are non-primary neuPrint ROIs in Fish2. Synapse/contact fetching therefore uses `SynapseCriteria(rois=..., primary_only=False)`, with batched body queries to avoid failures on large pulls.

In [ ]:
# ============================================================
# 6. QC soma-in-Raphe-superior dataset and choose body IDs for NBLAST
# ============================================================

datasets = {}

if RUN_RAPHE_SUPERIOR_SOMA_IN_ROI:
    sup_qc = build_qc_table(RAPHE_SUPERIOR_SOMA_IN_ROI_RAW_SWC_DIR, "raphe_superior_soma_in_roi")
    sup_pass = set(sup_qc.loc[sup_qc["qc_pass"], "bodyId"].dropna().astype(int)) if not sup_qc.empty else set()
    raphe_superior_soma_in_roi_ids_qc = [int(b) for b in raphe_superior_soma_in_roi_ids if int(b) in sup_pass]
    datasets["raphe_superior_soma_in_roi"] = {
        "label": "Raphe superior soma-in-ROI",
        "raw_swc_dir": RAPHE_SUPERIOR_SOMA_IN_ROI_RAW_SWC_DIR,
        "qc": sup_qc,
        "body_ids": raphe_superior_soma_in_roi_ids_qc,
    }
    print(f"raphe_superior_soma_in_roi: {len(raphe_superior_soma_in_roi_ids_qc)} QC-passing IDs")

if not datasets:
    raise RuntimeError("No datasets were prepared for clustering.")

display(pd.DataFrame([{
    "dataset": k,
    "label": v["label"],
    "qc_passing_ids": len(v["body_ids"]),
    "swc_dir": str(v["raw_swc_dir"]),
} for k, v in datasets.items()]))


In [ ]:
# ============================================================
# 7. NBLAST with fishFuncEM, using the v9.9 working pattern
# ============================================================

NBLAST_CACHE_DIR = OUT_ROOT / "nblast_cache"
NBLAST_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _ids_from_neurons_or_ids(neurons_or_ids) -> list[int]:
    if neurons_or_ids is None:
        return []
    if isinstance(neurons_or_ids, pd.DataFrame):
        if "bodyId" in neurons_or_ids.columns:
            return [int(x) for x in neurons_or_ids["bodyId"].dropna().astype(int).tolist()]
        return [int(x) for x in neurons_or_ids.index.tolist()]

    ids = []
    for x in list(neurons_or_ids):
        try:
            ids.append(int(x))
            continue
        except Exception:
            pass
        for attr in ["id", "bodyId", "name"]:
            if hasattr(x, attr):
                try:
                    ids.append(int(getattr(x, attr)))
                    break
                except Exception:
                    pass
    return list(dict.fromkeys(ids))


def _standardize_square_matrix(mat, ids: list[int] | None = None) -> pd.DataFrame:
    """
    Coerce fishFuncEM output into a square float DataFrame with integer body IDs.
    This avoids the previous bug where a matrix missing one ID killed the whole backend.
    """
    df = pd.DataFrame(mat).copy()

    if ids is not None and len(ids) == df.shape[0] == df.shape[1]:
        # Only use explicit IDs if labels do not already look like body IDs.
        try:
            current_index = [int(float(x)) for x in df.index]
            current_cols = [int(float(x)) for x in df.columns]
            overlap = len(set(current_index).intersection(set(map(int, ids))))
            if overlap < max(2, len(ids) // 2):
                df.index = list(map(int, ids))
                df.columns = list(map(int, ids))
        except Exception:
            df.index = list(map(int, ids))
            df.columns = list(map(int, ids))

    try:
        df.index = pd.Index([int(float(x)) for x in df.index], name="bodyId")
        df.columns = pd.Index([int(float(x)) for x in df.columns], name="bodyId")
    except Exception:
        if ids is not None and len(ids) == df.shape[0] == df.shape[1]:
            df.index = pd.Index(list(map(int, ids)), name="bodyId")
            df.columns = pd.Index(list(map(int, ids)), name="bodyId")
        else:
            raise ValueError("Could not infer integer body IDs for matrix rows/columns.")

    common = [int(x) for x in df.index if int(x) in set(map(int, df.columns))]
    df = df.loc[common, common]
    df = df.apply(pd.to_numeric, errors="coerce").astype(float)
    return df


def _symmetrize_similarity(sim: pd.DataFrame) -> pd.DataFrame:
    sim = _standardize_square_matrix(sim)
    ids = sim.index.astype(int).tolist()
    S = sim.to_numpy(float)

    finite = S[np.isfinite(S)]
    fill = float(np.nanmedian(finite)) if len(finite) else 0.0
    S = np.nan_to_num(S, nan=fill, posinf=fill, neginf=fill)
    S = (S + S.T) / 2

    try:
        np.fill_diagonal(S, 1.0)
    except Exception:
        pass

    return pd.DataFrame(S, index=ids, columns=ids)


def similarity_to_distance(sim: pd.DataFrame, mode: str = DISTANCE_MODE) -> pd.DataFrame:
    """
    Convert fishFuncEM NBLAST similarity to a symmetric distance matrix.
    Default mode 'rank' is robust to score-scale differences between neurons.
    """
    sim = _symmetrize_similarity(sim)
    ids = sim.index.astype(int).tolist()
    S = sim.to_numpy(float)
    n = S.shape[0]

    if mode == "raw_inverse":
        mx = np.nanmax(S)
        D = mx - S

    elif mode == "rank":
        D = np.zeros_like(S, dtype=float)
        for i in range(n):
            row = S[i].copy()
            finite = row[np.isfinite(row)]
            if len(finite) == 0:
                row[:] = 0
            else:
                row = np.where(np.isfinite(row), row, np.nanmin(finite))

            order = np.argsort(row)  # low similarity first
            ranks = np.empty(n, dtype=float)
            ranks[order] = np.linspace(1, 0, n)  # high similarity -> small distance
            D[i] = ranks

        D = (D + D.T) / 2

    else:
        raise ValueError(f"Unknown DISTANCE_MODE: {mode!r}. Use 'rank' or 'raw_inverse'.")

    D = np.asarray(D, dtype=float)
    finiteD = D[np.isfinite(D)]
    fill = float(np.nanmax(finiteD)) if len(finiteD) else 1.0
    D = np.nan_to_num(D, nan=fill, posinf=fill, neginf=0.0)
    D = np.maximum(D, 0)
    D = (D + D.T) / 2
    np.fill_diagonal(D, 0.0)

    return pd.DataFrame(D, index=ids, columns=ids)


def _parse_missing_ids_from_keyerror(exc: Exception) -> list[int]:
    """
    fishFuncEM can fail inside compute_nblast with:
        KeyError('[101719069] not in index')
    In that case, drop the missing requested ID and retry.
    """
    text = repr(exc)
    ids = re.findall(r"\b\d{6,}\b", text)
    return sorted({int(x) for x in ids})


def _run_fishfuncem_once(requested_ids: list[int], dataset_key: str):
    """
    One fishFuncEM attempt, copied from the working v9.9 pattern:
        NeuprintServer -> get_custom_neuron_list -> FishNeuronMorph -> compute_nblast(mirror=True)
    """
    configure_token_env_for_neuprint_and_fishfuncem()

    nps = NeuprintServer()
    neuron_df = nps.get_custom_neuron_list(requested_ids)
    neuron_df = neuron_df.copy()

    morph = FishNeuronMorph(neuron_df, nps)
    scores = morph.compute_nblast(mirror=FISHFUNCEM_MIRROR)

    return pd.DataFrame(scores).copy(), neuron_df


def run_fishfuncem_nblast(body_ids, dataset_key: str, mirror: bool = True):
    """
    Cached robust fishFuncEM NBLAST for one dataset.

    - Uses only the QC-passing body IDs supplied by the dataset cell.
    - Saves raw similarity immediately.
    - Saves distance matrix separately.
    - If a fishFuncEM KeyError reports a missing body ID, drops it and retries.
    """
    requested_original = _ids_from_neurons_or_ids(body_ids)
    requested_original = list(dict.fromkeys(int(x) for x in requested_original))

    if len(requested_original) < 2:
        print(f"{dataset_key}: fewer than 2 IDs for fishFuncEM.")
        return pd.DataFrame(), pd.DataFrame()

    cache_dir = NBLAST_CACHE_DIR / dataset_key
    cache_dir.mkdir(parents=True, exist_ok=True)

    sim_pkl = cache_dir / f"{dataset_key}_fishfuncem_similarity.pkl"
    dist_pkl = cache_dir / f"{dataset_key}_fishfuncem_distance_{DISTANCE_MODE}.pkl"
    sim_csv = cache_dir / f"{dataset_key}_fishfuncem_similarity.csv"
    dist_csv = cache_dir / f"{dataset_key}_fishfuncem_distance_{DISTANCE_MODE}.csv"
    missing_csv = cache_dir / f"{dataset_key}_fishfuncem_missing_ids.csv"
    retry_csv = cache_dir / f"{dataset_key}_fishfuncem_retry_log.csv"

    if (not FORCE_RECOMPUTE_NBLAST) and sim_pkl.exists() and dist_pkl.exists():
        sim = _standardize_square_matrix(pd.read_pickle(sim_pkl))
        dist = _standardize_square_matrix(pd.read_pickle(dist_pkl))
        print(f"Loaded cached fishFuncEM NBLAST for {dataset_key}: {sim.shape}")
        return sim, dist

    requested = list(requested_original)
    dropped = []
    retry_rows = []
    sim = None
    neuron_df = None
    last_error = None

    max_rounds = min(20, max(1, len(requested) - 1))

    print(f"Running fishFuncEM NBLAST for {dataset_key}: n={len(requested)}, mirror={mirror}")

    for attempt in range(1, max_rounds + 1):
        try:
            print(f"  attempt {attempt}: {len(requested)} requested IDs")
            sim, neuron_df = _run_fishfuncem_once(requested, dataset_key)

            retry_rows.append({
                "attempt": attempt,
                "n_requested": len(requested),
                "status": "success",
                "dropped_this_attempt": "",
                "error": "",
            })
            break

        except KeyError as e:
            last_error = e
            missing = [bid for bid in _parse_missing_ids_from_keyerror(e) if bid in requested]

            retry_rows.append({
                "attempt": attempt,
                "n_requested": len(requested),
                "status": "retry_after_keyerror",
                "dropped_this_attempt": ",".join(map(str, missing)),
                "error": repr(e),
            })

            if not missing:
                print("fishFuncEM KeyError did not contain a requested body ID.")
                traceback.print_exc(limit=2)
                raise

            print(f"  fishFuncEM missing-ID KeyError. Dropping and retrying: {missing}")
            dropped.extend(missing)
            requested = [bid for bid in requested if bid not in set(missing)]

            if len(requested) < 2:
                raise RuntimeError("Too many IDs dropped; fewer than 2 remain.") from e

        except Exception as e:
            last_error = e
            retry_rows.append({
                "attempt": attempt,
                "n_requested": len(requested),
                "status": "failed_other_error",
                "dropped_this_attempt": "",
                "error": repr(e),
            })
            print("fishFuncEM failed with a non-missing-ID error.")
            traceback.print_exc(limit=2)
            raise

    else:
        raise RuntimeError(f"fishFuncEM failed after retries. Last error: {repr(last_error)}")

    pd.DataFrame(retry_rows).to_csv(retry_csv, index=False)
    if dropped:
        pd.DataFrame({"bodyId": sorted(set(dropped)), "reason": "dropped_after_fishfuncem_keyerror"}).to_csv(missing_csv, index=False)
        print(f"Saved dropped-ID log: {missing_csv}")

    if neuron_df is not None:
        neuron_df.to_csv(cache_dir / f"{dataset_key}_fishfuncem_neuron_df_final.csv", index=False)

    # Standardize without assuming every requested ID survived.
    try:
        sim = _standardize_square_matrix(sim)
    except Exception:
        sim = _standardize_square_matrix(sim, ids=requested)

    axis_ids = set(map(int, sim.index)).intersection(set(map(int, sim.columns)))
    valid_ids = [bid for bid in requested if bid in axis_ids]

    missing_after = sorted(set(requested_original) - set(valid_ids))
    if missing_after:
        pd.DataFrame({"bodyId": missing_after, "reason": "missing_or_dropped_from_fishfuncem_matrix"}).to_csv(missing_csv, index=False)
        print(f"{dataset_key}: fishFuncEM continuing with {len(valid_ids)} / {len(requested_original)} IDs.")
        print(f"Saved missing-ID log: {missing_csv}")

    if len(valid_ids) < 2:
        raise RuntimeError(f"{dataset_key}: fishFuncEM returned fewer than 2 valid IDs.")

    sim = sim.loc[valid_ids, valid_ids]
    sim = _symmetrize_similarity(sim)

    dist = similarity_to_distance(sim, DISTANCE_MODE)

    sim.to_pickle(sim_pkl)
    dist.to_pickle(dist_pkl)
    sim.to_csv(sim_csv)
    dist.to_csv(dist_csv)

    print(f"Saved fishFuncEM NBLAST cache for {dataset_key}: {sim.shape}")
    return sim, dist


for key, ds in datasets.items():
    ids = ds.get("body_ids", [])
    if len(ids) < 2:
        print(f"Skipping {key}: fewer than 2 QC-passing IDs")
        continue

    sim, dist = run_fishfuncem_nblast(ids, key, mirror=FISHFUNCEM_MIRROR)
    ds["sim"] = sim
    ds["dist"] = dist

print("\nNBLAST complete.")
for key, ds in datasets.items():
    print(f"  {key}: sim={ds.get('sim', pd.DataFrame()).shape}, dist={ds.get('dist', pd.DataFrame()).shape}")


In [ ]:
# ============================================================
# 8. Clustering + embeddings
# ============================================================

def linkage_from_distance(dist):
    arr = np.asarray(dist, dtype=float)
    arr = (arr + arr.T) / 2
    np.fill_diagonal(arr, 0)
    arr = np.maximum(arr, 0)
    return linkage(squareform(arr, checks=False), method="average")


def cluster_sweep(dist, dataset_key):
    Z = linkage_from_distance(dist)
    heights = Z[:, 2]
    # Sweep many meaningful height quantiles, not tiny absolute values.
    qs = np.linspace(0.35, 0.95, 49)
    cuts = sorted(set(float(np.quantile(heights, q)) for q in qs if np.isfinite(np.quantile(heights, q))))
    rows = []
    n = dist.shape[0]
    labels_by_cut = {}
    for cut in cuts:
        labs = fcluster(Z, t=cut, criterion="distance")
        sizes = pd.Series(labs).value_counts().sort_values(ascending=False)
        n_clusters = int(len(sizes))
        n_singletons = int((sizes == 1).sum())
        n_real = int((sizes >= AUTO_MIN_MULTI_CLUSTER_SIZE).sum())
        singleton_frac = n_singletons / n
        largest = int(sizes.iloc[0]) if len(sizes) else 0
        # Coarse biological preference score.
        if n_real < AUTO_MIN_REAL_CLUSTERS:
            real_penalty = (AUTO_MIN_REAL_CLUSTERS - n_real) * 2.0
        elif n_real > AUTO_MAX_REAL_CLUSTERS:
            real_penalty = (n_real - AUTO_MAX_REAL_CLUSTERS) * 1.5
        else:
            real_penalty = 0
        singleton_penalty = max(0, singleton_frac - AUTO_MAX_SINGLETON_FRAC) * 8
        tiny_penalty = (n_clusters - n_real) / max(1, n) * 2
        score = real_penalty + singleton_penalty + tiny_penalty
        rows.append({
            "dataset": dataset_key,
            "cut": cut,
            "n_cells": n,
            "n_clusters_total": n_clusters,
            "n_real_clusters": n_real,
            "n_singletons": n_singletons,
            "singleton_frac": singleton_frac,
            "largest_cluster_size": largest,
            "auto_score": score,
            "cluster_sizes": ",".join(map(str, sizes.tolist()[:50])),
        })
        labels_by_cut[cut] = labs
    sweep = pd.DataFrame(rows).sort_values(["auto_score", "n_real_clusters", "singleton_frac"])
    chosen = sweep.iloc[0]
    chosen_cut = float(chosen["cut"])
    labs = labels_by_cut[chosen_cut]
    # Renumber clusters by descending size for readability.
    size_order = pd.Series(labs).value_counts().index.tolist()
    remap = {old: i+1 for i, old in enumerate(size_order)}
    labs2 = np.array([remap[x] for x in labs], dtype=int)
    assignments = pd.DataFrame({"bodyId": [int(x) for x in dist.index], "cluster": labs2})
    sizes = assignments["cluster"].value_counts().sort_index()
    assignments["cluster_size"] = assignments["cluster"].map(sizes)
    assignments["cluster_name"] = assignments["cluster"].map(lambda x: f"{dataset_key}_C{int(x):02d}")
    sweep.to_csv(OUT_ROOT / f"{dataset_key}_cluster_sweep.csv", index=False)
    assignments.to_csv(OUT_ROOT / f"{dataset_key}_cluster_assignments.csv", index=False)
    print(f"{dataset_key}: chosen cut={chosen_cut:.4f}; real clusters={int(chosen['n_real_clusters'])}; total clusters={int(chosen['n_clusters_total'])}")
    return Z, sweep, assignments, chosen


def compute_embeddings(dist, dataset_key):
    arr = np.asarray(dist, dtype=float)
    ids = [int(x) for x in dist.index]
    emb = pd.DataFrame({"bodyId": ids})
    mds = MDS(n_components=2, dissimilarity="precomputed", random_state=RANDOM_SEED, normalized_stress="auto")
    xy = mds.fit_transform(arr)
    emb["mds1"] = xy[:, 0]; emb["mds2"] = xy[:, 1]
    if HAS_UMAP:
        reducer = umap.UMAP(n_neighbors=min(UMAP_N_NEIGHBORS, max(2, len(ids)-1)), min_dist=UMAP_MIN_DIST, metric="precomputed", random_state=RANDOM_SEED)
        uv = reducer.fit_transform(arr)
        emb["umap1"] = uv[:, 0]; emb["umap2"] = uv[:, 1]
    emb.to_csv(OUT_ROOT / f"{dataset_key}_embeddings.csv", index=False)
    return emb

for key, ds in datasets.items():
    if "dist" not in ds:
        continue
    Z, sweep, assignments, chosen = cluster_sweep(ds["dist"], key)
    emb = compute_embeddings(ds["dist"], key)
    ds.update({"Z": Z, "sweep": sweep, "assignments": assignments, "chosen": chosen, "embeddings": emb})

In [ ]:
# ============================================================
# 9. Save UMAP/MDS figures and summary cluster-count figure
# ============================================================

FIG_DIR = OUT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def savefig_all(fig, path_base):
    path_base = Path(path_base)
    if SAVE_PNG: fig.savefig(path_base.with_suffix(".png"), dpi=FIG_DPI, bbox_inches="tight")
    if SAVE_PDF: fig.savefig(path_base.with_suffix(".pdf"), bbox_inches="tight")
    if SAVE_SVG: fig.savefig(path_base.with_suffix(".svg"), bbox_inches="tight")


def plot_embedding(ds, dataset_key, method="umap"):
    emb = ds["embeddings"].merge(ds["assignments"], on="bodyId", how="left")
    xcol, ycol = ("umap1", "umap2") if method == "umap" else ("mds1", "mds2")
    if xcol not in emb.columns:
        return
    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=FIG_DPI)
    sc = ax.scatter(emb[xcol], emb[ycol], c=emb["cluster"], s=35, cmap="tab20", edgecolor="none", alpha=0.9)
    ax.set_title(f"{ds['label']} | {method.upper()} | {emb['cluster'].nunique()} clusters")
    ax.set_xlabel(xcol); ax.set_ylabel(ycol)
    ax.set_aspect("equal", adjustable="datalim")
    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("cluster")
    fig.tight_layout()
    savefig_all(fig, FIG_DIR / f"{dataset_key}_{method}_clusters")
    display(fig); plt.close(fig)

for key, ds in datasets.items():
    if "embeddings" in ds:
        plot_embedding(ds, key, "umap")
        plot_embedding(ds, key, "mds")

# Summary figure comparing datasets.
summary_rows = []
for key, ds in datasets.items():
    if "assignments" not in ds:
        continue
    a = ds["assignments"]
    sizes = a["cluster"].value_counts()
    summary_rows.append({
        "dataset": ds["label"],
        "n_cells_clustered": len(a),
        "n_clusters_total": int(sizes.size),
        "n_multi_neuron_clusters": int((sizes >= 2).sum()),
        "n_singletons": int((sizes == 1).sum()),
        "largest_cluster": int(sizes.max()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_ROOT / "summary_cluster_counts.csv", index=False)
display(summary_df)

if not summary_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=FIG_DPI)
    x = np.arange(len(summary_df))
    w = 0.25
    ax.bar(x - w, summary_df["n_clusters_total"], width=w, label="total clusters")
    ax.bar(x, summary_df["n_multi_neuron_clusters"], width=w, label="multi-neuron clusters")
    ax.bar(x + w, summary_df["n_singletons"], width=w, label="singletons")
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["dataset"], rotation=20, ha="right")
    ax.set_ylabel("count")
    ax.set_title("Morphology clustering summary")
    ax.legend(frameon=False)
    fig.tight_layout()
    savefig_all(fig, FIG_DIR / "summary_cluster_counts_selected_vs_raphe_superior")
    display(fig); plt.close(fig)

In [ ]:
# ============================================================
# 10. Excel-friendly cluster ID exports
# ============================================================

EXPORT_DIR = OUT_ROOT / "cluster_id_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def export_cluster_ids(ds, dataset_key):
    if "assignments" not in ds:
        return
    a = ds["assignments"].copy().sort_values(["cluster", "bodyId"])
    a["dataset"] = dataset_key
    a["dataset_label"] = ds["label"]
    per_neuron_path = EXPORT_DIR / f"{dataset_key}_per_neuron_cluster_assignments.csv"
    a.to_csv(per_neuron_path, index=False)
    rows = []
    for cl, sub in a.groupby("cluster"):
        ids = sub["bodyId"].astype(int).tolist()
        rows.append({
            "dataset": dataset_key,
            "dataset_label": ds["label"],
            "cluster": int(cl),
            "cluster_name": f"{dataset_key}_C{int(cl):02d}",
            "n_cells": len(ids),
            "bodyIds_comma": ",".join(map(str, ids)),
            "bodyIds_semicolon": ";".join(map(str, ids)),
        })
    cluster_df = pd.DataFrame(rows).sort_values(["n_cells", "cluster"], ascending=[False, True])
    cluster_path = EXPORT_DIR / f"{dataset_key}_cluster_id_lists.csv"
    cluster_df.to_csv(cluster_path, index=False)
    print(f"Saved {per_neuron_path}")
    print(f"Saved {cluster_path}")
    if HAS_OPENPYXL:
        xlsx = EXPORT_DIR / f"{dataset_key}_cluster_ids.xlsx"
        with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
            a.to_excel(writer, sheet_name="per_neuron", index=False)
            cluster_df.to_excel(writer, sheet_name="per_cluster", index=False)
        print(f"Saved {xlsx}")
    return cluster_df

all_cluster_summaries = []
for key, ds in datasets.items():
    x = export_cluster_ids(ds, key)
    if x is not None:
        all_cluster_summaries.append(x)
if all_cluster_summaries:
    combined = pd.concat(all_cluster_summaries, ignore_index=True)
    combined.to_csv(EXPORT_DIR / "combined_cluster_id_lists.csv", index=False)
    display(combined)

In [ ]:
# ============================================================
# 11. Brain shell from neuPrint ROI meshes + rendering helpers
# ============================================================

MESH_DIR = OUT_ROOT / "neuprint_roi_meshes"
SHELL_DIR = OUT_ROOT / "outer_brain_shell"
RENDER_DIR = OUT_ROOT / "cluster_renderings"
for p in [MESH_DIR, SHELL_DIR, RENDER_DIR]: p.mkdir(parents=True, exist_ok=True)


def fetch_all_roi_meshes(client, max_rois=None):
    roi_names = sorted(client.meta["roiInfo"].keys())
    if max_rois is not None:
        roi_names = roi_names[:max_rois]
    paths = []
    for roi in tqdm(roi_names, desc="Fetch ROI meshes"):
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", roi)
        out = MESH_DIR / f"{safe}.obj"
        if out.exists():
            paths.append(out); continue
        try:
            client.fetch_roi_mesh(roi, export_path=str(out))
            paths.append(out)
        except Exception:
            pass
    print(f"ROI meshes available: {len(paths)}")
    return paths


def read_obj_vertices_faces(path):
    vertices=[]; faces=[]
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("v "):
                p=line.split(); vertices.append([float(p[1]), float(p[2]), float(p[3])])
            elif line.startswith("f "):
                face=[]
                for token in line.split()[1:]:
                    face.append(int(token.split('/')[0])-1)
                if len(face)>=3: faces.append(face[:3])
    return np.asarray(vertices, dtype=float), np.asarray(faces, dtype=int) if faces else np.zeros((0,3), dtype=int)


def build_outer_shell_from_roi_meshes(force=False, voxel_size=None, max_points=300000):
    shell_npz = SHELL_DIR / "outer_brain_shell_voxel_union.npz"
    if shell_npz.exists() and not force:
        data = np.load(shell_npz)
        return data["vertices"], data["faces"]
    mesh_paths = fetch_all_roi_meshes(c)
    pts=[]
    for p in mesh_paths:
        v, f = read_obj_vertices_faces(p)
        if len(v): pts.append(v)
    if not pts:
        print("No ROI mesh vertices found; shell unavailable.")
        return None, None
    pts = np.vstack(pts)
    pts = pts[np.isfinite(pts).all(axis=1)]
    if len(pts) > max_points:
        rng = np.random.default_rng(RANDOM_SEED)
        pts = pts[rng.choice(len(pts), max_points, replace=False)]
    mn = pts.min(axis=0); mx = pts.max(axis=0)
    span = mx - mn
    if voxel_size is None:
        voxel_size = float(np.max(span) / 160.0)
    idx = np.floor((pts - mn) / voxel_size).astype(int)
    shape = idx.max(axis=0) + 3
    # keep grid manageable
    if np.prod(shape) > 220**3:
        voxel_size = float(np.max(span) / 180.0)
        idx = np.floor((pts - mn) / voxel_size).astype(int)
        shape = idx.max(axis=0) + 3
    vol = np.zeros(shape, dtype=bool)
    idx = np.clip(idx, 0, np.array(shape)-1)
    vol[idx[:,0], idx[:,1], idx[:,2]] = True
    # Fill along all 2D slices to approximate union volume.
    for z in range(vol.shape[2]): vol[:,:,z] = binary_fill_holes(vol[:,:,z])
    for y in range(vol.shape[1]): vol[:,y,:] = binary_fill_holes(vol[:,y,:])
    for x in range(vol.shape[0]): vol[x,:,:] = binary_fill_holes(vol[x,:,:])
    verts, faces, normals, values = marching_cubes(vol.astype(float), level=0.5)
    verts_world = verts * voxel_size + mn
    np.savez_compressed(shell_npz, vertices=verts_world, faces=faces.astype(int), voxel_size=voxel_size, origin=mn)
    print(f"Saved outer shell: {shell_npz} | vertices={len(verts_world)} faces={len(faces)}")
    return verts_world, faces.astype(int)

outer_shell_vertices, outer_shell_faces = build_outer_shell_from_roi_meshes(force=False)


# ------------------------------------------------------------
# Clean brain shell mesh: remove disconnected ROI/mask blobs
# ------------------------------------------------------------
# The raw shell built from all neuPrint ROI meshes can include small disconnected
# components outside the brain. For figure-ready rendering we keep only the
# largest connected mesh component, which gives a cleaner translucent brain
# outline like the reference figure.
SHELL_KEEP_LARGEST_COMPONENT_ONLY = True
SHELL_MIN_COMPONENT_FACE_FRACTION = 0.02


def clean_outer_shell_mesh_largest_component(vertices, faces, min_face_fraction=SHELL_MIN_COMPONENT_FACE_FRACTION):
    """
    Keep only the largest connected component of a triangular mesh.

    Connectivity is defined by faces sharing vertices. This removes small
    disconnected ROI blobs / masks outside the main brain shell.
    """
    if vertices is None or faces is None:
        return vertices, faces, {"status": "missing"}

    vertices = np.asarray(vertices, dtype=float)
    faces = np.asarray(faces, dtype=int)

    if len(vertices) == 0 or len(faces) == 0:
        return vertices, faces, {"status": "empty"}

    from collections import defaultdict, deque

    vertex_to_faces = defaultdict(list)
    for fi, tri in enumerate(faces):
        for v in tri:
            vertex_to_faces[int(v)].append(fi)

    seen = np.zeros(len(faces), dtype=bool)
    components = []

    for start in range(len(faces)):
        if seen[start]:
            continue
        q = deque([start])
        seen[start] = True
        comp = []
        while q:
            fi = q.popleft()
            comp.append(fi)
            for v in faces[fi]:
                for nb in vertex_to_faces[int(v)]:
                    if not seen[nb]:
                        seen[nb] = True
                        q.append(nb)
        components.append(np.asarray(comp, dtype=int))

    components = sorted(components, key=len, reverse=True)
    largest = components[0]
    min_faces = max(1, int(len(faces) * float(min_face_fraction)))

    # Default: largest component only. This is what removes the little blobs.
    keep_faces = largest

    kept_faces = faces[keep_faces]
    used_vertices = np.unique(kept_faces.ravel())
    old_to_new = {int(old): i for i, old in enumerate(used_vertices)}
    remapped_faces = np.vectorize(old_to_new.get)(kept_faces).astype(int)
    cleaned_vertices = vertices[used_vertices]

    info = {
        "status": "cleaned",
        "n_components": len(components),
        "original_vertices": int(len(vertices)),
        "original_faces": int(len(faces)),
        "kept_vertices": int(len(cleaned_vertices)),
        "kept_faces": int(len(remapped_faces)),
        "largest_component_faces": int(len(largest)),
        "min_faces_threshold": int(min_faces),
        "removed_faces": int(len(faces) - len(remapped_faces)),
        "component_face_counts_top10": [int(len(c)) for c in components[:10]],
    }

    print("Brain shell cleaning:")
    print(f"  components found: {info['n_components']}")
    print(f"  original faces: {info['original_faces']:,}")
    print(f"  kept faces:     {info['kept_faces']:,}")
    print(f"  removed faces:  {info['removed_faces']:,}")
    print(f"  top component face counts: {info['component_face_counts_top10']}")

    return cleaned_vertices, remapped_faces, info


if SHELL_KEEP_LARGEST_COMPONENT_ONLY and outer_shell_vertices is not None and outer_shell_faces is not None:
    outer_shell_vertices, outer_shell_faces, SHELL_CLEANING_INFO = clean_outer_shell_mesh_largest_component(
        outer_shell_vertices,
        outer_shell_faces,
    )
else:
    SHELL_CLEANING_INFO = {"status": "not_cleaned"}

print("Shell:", None if outer_shell_vertices is None else outer_shell_vertices.shape)

In [ ]:
# ============================================================
# 12. Cluster rendering: static nice figures + interactive 3D
# ============================================================

def swc_segments_for_body(body_id, swc_dir):
    path = Path(swc_dir) / f"{int(body_id)}.swc"
    if not path.exists():
        # try any filename containing ID
        matches = list(Path(swc_dir).glob(f"*{int(body_id)}*.swc"))
        if not matches: return [], None
        path = matches[0]
    df = read_swc(path)
    coords = df.set_index("node_id")[["x","y","z"]]
    segs=[]
    for p,n in swc_edges(df):
        try:
            a=coords.loc[p].values; b=coords.loc[n].values
            segs.append((a,b))
        except Exception: pass
    soma = estimate_soma_from_swc(df)
    return segs, soma


def add_shell_trace(fig, alpha=SHELL_ALPHA):
    if outer_shell_vertices is None or outer_shell_faces is None or len(outer_shell_vertices)==0:
        return fig
    v=outer_shell_vertices; f=outer_shell_faces
    fig.add_trace(go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2],
        i=f[:,0], j=f[:,1], k=f[:,2],
        opacity=alpha, color="rgb(210, 215, 220)", name="outer brain shell", showscale=False,
        hoverinfo="skip"
    ))
    return fig


def add_neuron_traces(fig, body_ids, swc_dir, name_prefix="cell", width=CELL_LINE_WIDTH, show_somas=True):
    for bid in body_ids:
        segs, soma = swc_segments_for_body(bid, swc_dir)
        if not segs: continue
        xs=[]; ys=[]; zs=[]
        for a,b in segs:
            xs += [a[0], b[0], None]; ys += [a[1], b[1], None]; zs += [a[2], b[2], None]
        fig.add_trace(go.Scatter3d(x=xs,y=ys,z=zs,mode="lines", line=dict(width=width), name=str(bid), hovertext=str(bid)))
        if show_somas and soma is not None:
            fig.add_trace(go.Scatter3d(x=[soma[0]], y=[soma[1]], z=[soma[2]], mode="markers+text", marker=dict(size=4), text=[str(bid)], textposition="top center", name=f"soma {bid}", showlegend=False))
    return fig


def render_cluster_3d(ds, dataset_key, cluster, save_html=True, show_population_context=False):
    a = ds["assignments"]
    body_ids = a.loc[a["cluster"]==int(cluster), "bodyId"].astype(int).tolist()
    swc_dir = ds["raw_swc_dir"]
    fig = go.Figure()
    if SHOW_BRAIN_SHELL:
        add_shell_trace(fig, alpha=SHELL_ALPHA)
    if show_population_context:
        context_ids = a["bodyId"].astype(int).tolist()
        # draw context as very thin transparent lines by using fewer cells if huge
        for bid in context_ids[:200]:
            segs, soma = swc_segments_for_body(bid, swc_dir)
            xs=[]; ys=[]; zs=[]
            for p,q in segs:
                xs += [p[0], q[0], None]; ys += [p[1], q[1], None]; zs += [p[2], q[2], None]
            fig.add_trace(go.Scatter3d(x=xs,y=ys,z=zs,mode="lines", line=dict(width=1, color="rgba(0,0,0,0.07)"), showlegend=False, hoverinfo="skip"))
    add_neuron_traces(fig, body_ids[:MAX_CELLS_PER_CLUSTER_RENDER], swc_dir, show_somas=SHOW_SOMA_MARKERS)
    fig.update_layout(
        title=f"{ds['label']} | cluster {cluster} | n={len(body_ids)}<br>IDs: {', '.join(map(str, body_ids[:30]))}{'...' if len(body_ids)>30 else ''}",
        scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data"),
        width=950, height=850,
        legend=dict(itemsizing="constant")
    )
    if save_html and SAVE_INTERACTIVE_HTML:
        outdir = RENDER_DIR / dataset_key / "interactive_3d"
        outdir.mkdir(parents=True, exist_ok=True)
        html = outdir / f"{dataset_key}_cluster_{int(cluster):02d}_3d.html"
        fig.write_html(str(html))
        print(f"Saved HTML: {html}")
    return fig

# Create a few example HTML renders for non-singleton clusters.
if MAKE_CLUSTER_RENDERINGS:
    for key, ds in datasets.items():
        if "assignments" not in ds: continue
        sizes = ds["assignments"]["cluster"].value_counts().sort_values(ascending=False)
        clusters = [int(c) for c,s in sizes.items() if s >= 2]
        if MAX_CLUSTERS_TO_RENDER is not None:
            clusters = clusters[:MAX_CLUSTERS_TO_RENDER]
        for cl in clusters:
            fig = render_cluster_3d(ds, key, cl, save_html=True, show_population_context=False)
            # display only first few inline to avoid notebook clutter
            if cl in clusters[:2]:
                display(fig)

In [ ]:
# ============================================================
# 13. Interactive cluster selector widgets
# ============================================================

try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False


def launch_cluster_selector():
    if not HAS_WIDGETS:
        print("ipywidgets not available. Use render_cluster_3d(datasets['selected_raphe'], 'selected_raphe', cluster=1).")
        return
    dataset_options = [(ds["label"], key) for key, ds in datasets.items() if "assignments" in ds]
    dataset_dd = widgets.Dropdown(options=dataset_options, description="Dataset:")
    cluster_dd = widgets.Dropdown(description="Cluster:")
    show_context = widgets.Checkbox(value=False, description="Population context")
    show_somas = widgets.Checkbox(value=SHOW_SOMA_MARKERS, description="Somas")
    out = widgets.Output()
    
    def update_clusters(*args):
        key = dataset_dd.value
        ds = datasets[key]
        sizes = ds["assignments"]["cluster"].value_counts().sort_index()
        opts = [(f"C{int(c):02d} | n={int(s)}", int(c)) for c,s in sizes.items()]
        cluster_dd.options = opts
    
    def draw(*args):
        global SHOW_SOMA_MARKERS
        SHOW_SOMA_MARKERS = bool(show_somas.value)
        with out:
            clear_output(wait=True)
            key = dataset_dd.value
            cl = cluster_dd.value
            ds = datasets[key]
            ids = ds["assignments"].loc[ds["assignments"]["cluster"]==cl, "bodyId"].astype(int).tolist()
            print(f"{ds['label']} | cluster {cl} | n={len(ids)}")
            print("IDs:", ", ".join(map(str, ids)))
            display(pd.DataFrame({"bodyId": ids}))
            fig = render_cluster_3d(ds, key, cl, save_html=True, show_population_context=show_context.value)
            display(fig)
    
    dataset_dd.observe(update_clusters, names="value")
    cluster_dd.observe(draw, names="value")
    show_context.observe(draw, names="value")
    show_somas.observe(draw, names="value")
    update_clusters()
    display(widgets.VBox([widgets.HBox([dataset_dd, cluster_dd]), widgets.HBox([show_context, show_somas]), out]))
    draw()

launch_cluster_selector()

In [ ]:
# ============================================================
# 14. Check which cluster a list of cell IDs belongs to
# ============================================================

# Edit this list and rerun the cell.
CELLS_TO_CHECK = [
    # 100010951,
    # 100014398,
]

def check_cells_in_existing_clusters(cell_ids=None):
    """
    Search both completed datasets and report which cluster each cell belongs to.
    This does not compute new NBLAST; it only checks already clustered QC-passing cells.
    """
    if cell_ids is None:
        cell_ids = CELLS_TO_CHECK

    cell_ids = [int(x) for x in cell_ids]
    rows = []

    for bid in cell_ids:
        found = False

        for dataset_key, ds in datasets.items():
            if "assignments" not in ds:
                continue

            a = ds["assignments"].copy()
            hit = a.loc[a["bodyId"].astype(int) == int(bid)]

            if len(hit):
                for _, r in hit.iterrows():
                    rows.append({
                        "bodyId": int(bid),
                        "dataset": dataset_key,
                        "dataset_label": ds.get("label", dataset_key),
                        "cluster": int(r["cluster"]),
                        "cluster_name": f"{dataset_key}_C{int(r['cluster']):02d}",
                        "cluster_size": int(a.loc[a["cluster"] == r["cluster"], "bodyId"].nunique()),
                        "status": "clustered",
                    })
                found = True

        if not found:
            rows.append({
                "bodyId": int(bid),
                "dataset": "",
                "dataset_label": "",
                "cluster": np.nan,
                "cluster_name": "",
                "cluster_size": np.nan,
                "status": "not_found_in_clustered_QC_passing_cells",
            })

    out = pd.DataFrame(rows)
    display(out)

    path = OUT_ROOT / "checked_cell_cluster_membership.csv"
    out.to_csv(path, index=False)
    print(f"Saved: {path}")

    return out


checked_cells = check_cells_in_existing_clusters()


In [ ]:
# ============================================================
# 15. Optional: assign new neurons a new list of neurons to the closest existing cluster
# ============================================================

# Edit this list directly, or create data/new_cell_ids.csv with a bodyId column.
NEW_CELL_IDS = []

# Choose reference dataset for assignment.
ASSIGN_REFERENCE_DATASET = "selected_raphe"  # or "raphe_superior"


def load_new_cell_ids():
    ids = [int(x) for x in NEW_CELL_IDS]
    if NEW_CELL_IDS_CSV.exists():
        df = pd.read_csv(NEW_CELL_IDS_CSV)
        col = "bodyId" if "bodyId" in df.columns else df.columns[0]
        ids.extend(df[col].dropna().astype(int).tolist())
    # Also include any SWC filenames in skeletons/new_cells_raw_swc.
    ids.extend([body_id_from_path(p) for p in NEW_CELLS_RAW_SWC_DIR.glob("*.swc")])
    ids = sorted(set(int(x) for x in ids if x is not None))
    return ids


def assign_new_cells_to_existing_clusters(new_ids=None, reference_dataset=ASSIGN_REFERENCE_DATASET):
    if new_ids is None:
        new_ids = load_new_cell_ids()
    new_ids = [int(x) for x in new_ids]
    if not new_ids:
        print("No new cell IDs supplied. Edit NEW_CELL_IDS or create data/new_cell_ids.csv.")
        return pd.DataFrame()
    if reference_dataset not in datasets or "assignments" not in datasets[reference_dataset]:
        raise RuntimeError(f"Reference dataset {reference_dataset!r} not available/clustering not run.")
    ref = datasets[reference_dataset]
    ref_ids = ref["assignments"]["bodyId"].astype(int).tolist()
    combined_ids = list(dict.fromkeys(ref_ids + new_ids))
    print(f"Computing combined fishFuncEM NBLAST for assignment: {len(ref_ids)} reference + {len(new_ids)} new")
    sim, dist = run_fishfuncem_nblast(combined_ids, f"assignment_{reference_dataset}", mirror=FISHFUNCEM_MIRROR)
    assignments = ref["assignments"].set_index("bodyId")
    rows=[]
    for nid in new_ids:
        if nid not in dist.index:
            rows.append({"new_bodyId": nid, "assigned_cluster": np.nan, "assigned_cluster_name": "missing_from_nblast", "distance_to_cluster": np.nan, "nearest_ref_bodyId": np.nan})
            continue
        best = None
        for cl, sub in assignments.groupby("cluster"):
            cluster_ids = [int(x) for x in sub.index if int(x) in dist.columns]
            if not cluster_ids:
                continue
            dvals = dist.loc[nid, cluster_ids].astype(float)
            med = float(np.nanmedian(dvals))
            nearest = int(dvals.idxmin())
            min_d = float(dvals.min())
            score = med  # median distance to cluster is more stable than nearest-neighbor alone
            if best is None or score < best["distance_to_cluster"]:
                best = {"assigned_cluster": int(cl), "assigned_cluster_name": f"{reference_dataset}_C{int(cl):02d}", "distance_to_cluster": score, "nearest_ref_bodyId": nearest, "nearest_distance": min_d}
        row = {"new_bodyId": nid, **(best or {})}
        rows.append(row)
    result = pd.DataFrame(rows).sort_values(["assigned_cluster", "distance_to_cluster"])
    out = OUT_ROOT / f"new_cells_assigned_to_{reference_dataset}_clusters.csv"
    result.to_csv(out, index=False)
    print(f"Saved assignment table: {out}")
    display(result)
    return result

# Example:
# NEW_CELL_IDS = [123456789, 987654321]
# new_assignments = assign_new_cells_to_existing_clusters()

# v19-style plotting and figure-ready exports for the new soma-in-ROI run

The cells below reload the outputs just created above and expose the same plotting conveniences as v19: stable cluster colors, UMAP/MDS exports, three 3D cluster views, figure-ready projection panels, and a custom cluster export cell.

In [ ]:
# ============================================================
# 1. Imports + configuration
# ============================================================

from pathlib import Path
from collections import defaultdict
import os
import re
import json
import math
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
from tqdm import tqdm

# Plotly for interactive cluster rendering
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception as e:
    HAS_PLOTLY = False
    print("Plotly not available:", repr(e))

# Optional widgets for cluster selector
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception as e:
    HAS_WIDGETS = False
    print("ipywidgets not available:", repr(e))

# Optional neuPrint soma metadata fetching.
try:
    from neuprint import Client, fetch_neurons, NeuronCriteria as NC
    HAS_NEUPRINT = True
except Exception as e:
    HAS_NEUPRINT = False
    print("neuprint-python not available:", repr(e))

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
SKELETON_DIR = PROJECT_DIR / "skeletons"

# This should be the output folder from your previous working v14 notebook.
PREVIOUS_OUT_ROOT = PROJECT_DIR / "fish2_raphe_morphology_v20_rapheSuperior_somaInROI_outputs"

# This notebook writes only new/updated plots and tables here.
PLOT_OUT_ROOT = PREVIOUS_OUT_ROOT / "plotting_v19_figure_ready"
PLOT_OUT_ROOT.mkdir(parents=True, exist_ok=True)

SELECTED_RAW_SWC_DIR = SKELETON_DIR / "selected_raw_swc"
RAPHE_SUPERIOR_SOMA_IN_ROI_RAW_SWC_DIR = SKELETON_DIR / "raphe_superior_soma_in_roi_raw_swc"

# Dataset keys from v14.
DATASET_CONFIG = {
    "raphe_superior_soma_in_roi": {
        "label": "Raphe superior soma-in-ROI neurons",
        "raw_swc_dir": SKELETON_DIR / "raphe_superior_soma_in_roi_raw_swc",
    },
}

# Figure export settings.
FIG_DPI = 300
SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = True
SAVE_INTERACTIVE_HTML = True

# Rendering options.
SHOW_BRAIN_SHELL = True
SHELL_ALPHA = 0.055
CELL_LINE_WIDTH = 2.4  # Tune this: smaller values make crowded clusters easier to inspect
CONTEXT_LINE_WIDTH = 0.7
POPULATION_CONTEXT_ALPHA = 0.035
MAX_CONTEXT_CELLS = 250
MAX_CELLS_PER_CLUSTER_RENDER = 60

# v18 cluster-view modes.
# "all_three" displays three separate brains: skeletons-only, somas-only, and combined.
# Other options display one brain.
DEFAULT_CLUSTER_RENDER_MODE = "all_three"
CLUSTER_RENDER_MODE_OPTIONS = ["all_three", "skeletons_only", "somas_only", "combined"]
SAVE_EACH_RENDER_MODE_HTML = True


# Display-only orientation correction.
# This flips the shell, neurons, and somas together, only for plotting.
# It does not alter clusters, SWCs, UMAPs, or CSV exports.
RENDER_FLIP_X = False
RENDER_FLIP_Y = False
RENDER_FLIP_Z = True

# Real soma/cell-body rendering.
USE_NEUPRINT_SOMAS = True
NEUPRINT_SERVER = "https://neuprint-fish2.janelia.org"
NEUPRINT_DATASET = "fish2"
NEUPRINT_TOKEN_FALLBACK = ""  # Prefer env var NEUPRINT_TOKEN or local neuprint_token.txt
SOMA_LOCATION_PRIORITY = ["somaLocation", "tosomaLocation", "rootLocation"]
# Soma/cell-body display mode:
#   "swc_local_blob" = preferred: render local SWC nodes/edges around soma/root with radius-scaled markers.
#   "sphere"         = metadata somaLocation/somaRadius as a simple sphere.
#   "marker"         = metadata/root as a simple point.
SOMA_RENDER_MODE = "swc_radius_mesh"
SOMA_MARKER_SIZE = 5
SOMA_SPHERE_OPACITY = 0.9
SOMA_SPHERE_RADIUS_SCALE = 1.0
SOMA_MARKER_SIZE_MIN = 4
SOMA_MARKER_SIZE_MAX = 12
SOMA_SPHERE_THETA_STEPS = 10
SOMA_SPHERE_PHI_STEPS = 8
FORCE_REFETCH_SOMAS = False

# Skeleton display settings.
# "lines" is fast and clean. "radius_tubes" is experimental and much heavier.
CELL_RENDER_MODE = "lines"
CELL_LINE_ALPHA = 0.95
CELL_TUBE_RADIUS_SCALE = 0.45
CELL_TUBE_MIN_RADIUS = 8.0
CELL_TUBE_MAX_RADIUS = 35.0
CELL_TUBE_MAX_EDGES_PER_NEURON = 1200
CELL_TUBE_SIDES = 6

# Soma/cell-body display settings.
# "swc_radius_mesh" is default: render a small surface mesh from the local SWC
# nodes/radii around the neuPrint soma/root anchor. This is the most realistic
# soma geometry available from neuPrint/SWC without external segmentation meshes.
# "local_body_mesh" tries body_meshes/<bodyId>.obj/.ply and clips near soma.
# "marker" and "sphere" are simple fallbacks.
SOMA_RADIUS_MESH_GRAPH_HOPS = 5
SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS = 650.0
SOMA_RADIUS_MESH_MAX_NODES = 22
SOMA_RADIUS_MESH_MIN_NODES = 4
SOMA_RADIUS_MESH_RADIUS_SCALE = 1.15
SOMA_RADIUS_MESH_MIN_RADIUS = 25.0
SOMA_RADIUS_MESH_MAX_RADIUS = 230.0
SOMA_RADIUS_MESH_SPHERE_THETA = 9
SOMA_RADIUS_MESH_SPHERE_PHI = 7
SOMA_RADIUS_MESH_CYLINDER_SIDES = 9
SOMA_RADIUS_MESH_OPACITY = 0.92

# Optional true segmentation/body mesh hook.
# If you export per-body meshes later, place them in body_meshes/ as <bodyId>.obj or <bodyId>.ply.
# The notebook will clip vertices/faces near somaLocation/root and render that soma-region mesh.
LOCAL_BODY_MESH_DIR = PROJECT_DIR / "body_meshes"
PREFER_LOCAL_BODY_MESH_SOMA = True
LOCAL_BODY_MESH_SOMA_CLIP_RADIUS = 1200.0
LOCAL_BODY_MESH_MAX_FACES = 8000
LOCAL_BODY_MESH_OPACITY = 0.82

# Cluster colors: every cell and soma in a displayed cluster uses the same color.
CLUSTER_COLOR_PALETTE = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173",
    "#3182bd", "#31a354", "#756bb1", "#636363", "#e6550d",
]

# Output folders.
FIG_DIR = PLOT_OUT_ROOT / "figures"
RENDER_DIR = PLOT_OUT_ROOT / "cluster_renderings"
EXPORT_DIR = PLOT_OUT_ROOT / "cluster_id_exports"
SOMA_DIR = PLOT_OUT_ROOT / "neuprint_soma_metadata"
for p in [FIG_DIR, RENDER_DIR, EXPORT_DIR, SOMA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Previous outputs:", PREVIOUS_OUT_ROOT)
print("New plot-only outputs:", PLOT_OUT_ROOT)
print("Render flips: X=", RENDER_FLIP_X, "Y=", RENDER_FLIP_Y, "Z=", RENDER_FLIP_Z)


In [ ]:
# ============================================================
# 2. Load previous v14 clustering outputs
# ============================================================

def require_file(path, description="file"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            "This plotting-only notebook expects that the v14 notebook has already completed."
        )
    return path


def load_dataset_from_previous_outputs(dataset_key, cfg):
    ds = dict(cfg)
    # Prefer the root assignment file, then fall back to cluster_id_exports.
    assign_path = PREVIOUS_OUT_ROOT / f"{dataset_key}_cluster_assignments.csv"
    if not assign_path.exists():
        assign_path = PREVIOUS_OUT_ROOT / "cluster_id_exports" / f"{dataset_key}_per_neuron_cluster_assignments.csv"
    require_file(assign_path, f"{dataset_key} assignments")
    assignments = pd.read_csv(assign_path)
    assignments["bodyId"] = assignments["bodyId"].astype(int)
    assignments["cluster"] = assignments["cluster"].astype(int)
    if "cluster_size" not in assignments.columns:
        sizes = assignments["cluster"].value_counts()
        assignments["cluster_size"] = assignments["cluster"].map(sizes).astype(int)
    if "cluster_name" not in assignments.columns:
        assignments["cluster_name"] = assignments["cluster"].map(lambda x: f"{dataset_key}_C{int(x):02d}")
    ds["assignments"] = assignments
    ds["body_ids"] = assignments["bodyId"].astype(int).tolist()
    ds["assignment_path"] = assign_path

    emb_path = PREVIOUS_OUT_ROOT / f"{dataset_key}_embeddings.csv"
    if emb_path.exists():
        emb = pd.read_csv(emb_path)
        emb["bodyId"] = emb["bodyId"].astype(int)
        ds["embeddings"] = emb
        ds["embedding_path"] = emb_path
    else:
        print(f"No embedding file found for {dataset_key}: {emb_path}")

    sweep_path = PREVIOUS_OUT_ROOT / f"{dataset_key}_cluster_sweep.csv"
    if sweep_path.exists():
        ds["sweep"] = pd.read_csv(sweep_path)

    return ds


datasets = {}
for key, cfg in DATASET_CONFIG.items():
    try:
        datasets[key] = load_dataset_from_previous_outputs(key, cfg)
        a = datasets[key]["assignments"]
        print(f"Loaded {key}: {len(a)} cells, {a['cluster'].nunique()} clusters from {datasets[key]['assignment_path']}")
    except FileNotFoundError as e:
        print(str(e))

if not datasets:
    raise RuntimeError("No previous dataset outputs could be loaded.")

# Quick display of cluster sizes.
for key, ds in datasets.items():
    print("\n", "="*80)
    print(ds["label"])
    display(ds["assignments"].groupby("cluster").size().sort_values(ascending=False).to_frame("n_cells").head(30))


In [ ]:
# ============================================================
# 3. Re-save UMAP/MDS figures from previous embeddings
# ============================================================

import matplotlib.colors as mcolors


def savefig_all(fig, path_base):
    path_base = Path(path_base)
    if SAVE_PNG:
        fig.savefig(path_base.with_suffix(".png"), dpi=FIG_DPI, bbox_inches="tight")
    if SAVE_PDF:
        fig.savefig(path_base.with_suffix(".pdf"), bbox_inches="tight")
    if SAVE_SVG:
        fig.savefig(path_base.with_suffix(".svg"), bbox_inches="tight")


def get_cluster_color(dataset_key, cluster):
    """Stable cluster color used across UMAPs and 3D cluster renders."""
    cluster = int(cluster)
    return CLUSTER_COLOR_PALETTE[(cluster - 1) % len(CLUSTER_COLOR_PALETTE)]


def get_cluster_color_map(dataset_key, clusters):
    clusters = sorted([int(c) for c in pd.Series(clusters).dropna().unique()])
    return {cl: get_cluster_color(dataset_key, cl) for cl in clusters}


def plot_embedding_from_loaded(ds, dataset_key, method="umap", label_points=False):
    if "embeddings" not in ds:
        print(f"Skipping {dataset_key} {method}: no embeddings loaded.")
        return None
    emb = ds["embeddings"].merge(ds["assignments"], on="bodyId", how="left")
    xcol, ycol = ("umap1", "umap2") if method == "umap" else ("mds1", "mds2")
    if xcol not in emb.columns or ycol not in emb.columns:
        print(f"Skipping {dataset_key} {method}: columns {xcol}/{ycol} not found.")
        return None

    clusters = sorted(emb["cluster"].dropna().astype(int).unique())
    color_map = get_cluster_color_map(dataset_key, clusters)

    fig, ax = plt.subplots(figsize=(6.8, 5.8), dpi=FIG_DPI)
    for cl in clusters:
        sub = emb.loc[emb["cluster"].astype(int) == cl]
        ax.scatter(
            sub[xcol], sub[ycol],
            color=color_map[cl],
            s=46,
            alpha=0.92,
            edgecolor="white",
            linewidth=0.35,
            label=f"C{cl:02d} (n={len(sub)})",
        )
    if label_points and len(emb) <= 150:
        for _, r in emb.iterrows():
            ax.text(r[xcol], r[ycol], str(int(r["bodyId"])), fontsize=4, alpha=0.65)
    ax.set_title(f"{ds['label']} | {method.upper()} | {emb['cluster'].nunique()} clusters")
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.set_aspect("equal", adjustable="datalim")
    if len(clusters) <= 18:
        ax.legend(frameon=False, fontsize=6, markerscale=0.7, loc="best")
    fig.tight_layout()
    savefig_all(fig, FIG_DIR / f"{dataset_key}_{method}_clusters_plotonly_v16_clusterColors")
    display(fig)
    plt.close(fig)
    return emb

for key, ds in datasets.items():
    plot_embedding_from_loaded(ds, key, method="umap", label_points=False)
    plot_embedding_from_loaded(ds, key, method="mds", label_points=False)

# Summary figure from loaded assignments.
summary_rows = []
for key, ds in datasets.items():
    a = ds["assignments"]
    sizes = a["cluster"].value_counts()
    summary_rows.append({
        "dataset_key": key,
        "dataset": ds["label"],
        "n_cells_clustered": len(a),
        "n_clusters_total": int(sizes.size),
        "n_multi_neuron_clusters": int((sizes >= 2).sum()),
        "n_singletons": int((sizes == 1).sum()),
        "largest_cluster": int(sizes.max()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(PLOT_OUT_ROOT / "summary_cluster_counts_loaded_from_v14.csv", index=False)
display(summary_df)

if not summary_df.empty:
    fig, ax = plt.subplots(figsize=(7.4, 4.8), dpi=FIG_DPI)
    x = np.arange(len(summary_df))
    w = 0.25
    ax.bar(x - w, summary_df["n_clusters_total"], width=w, label="total clusters")
    ax.bar(x, summary_df["n_multi_neuron_clusters"], width=w, label="multi-neuron clusters")
    ax.bar(x + w, summary_df["n_singletons"], width=w, label="singletons")
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["dataset"], rotation=20, ha="right")
    ax.set_ylabel("Count")
    ax.set_title("Morphology clustering summary — loaded from v14")
    ax.legend(frameon=False)
    fig.tight_layout()
    savefig_all(fig, FIG_DIR / "summary_cluster_counts_loaded_from_v14")
    display(fig)
    plt.close(fig)


In [ ]:
# ============================================================
# 4. Excel-friendly cluster ID exports from loaded assignments
# ============================================================

try:
    import openpyxl  # noqa
    HAS_OPENPYXL = True
except Exception:
    HAS_OPENPYXL = False


def export_loaded_cluster_ids(ds, dataset_key):
    a = ds["assignments"].copy().sort_values(["cluster", "bodyId"])
    a["dataset"] = dataset_key
    a["dataset_label"] = ds["label"]
    a["cluster_name"] = a["cluster"].map(lambda x: f"{dataset_key}_C{int(x):02d}")
    sizes = a["cluster"].value_counts()
    a["cluster_size"] = a["cluster"].map(sizes).astype(int)

    per_neuron_path = EXPORT_DIR / f"{dataset_key}_per_neuron_cluster_assignments_loaded.csv"
    a.to_csv(per_neuron_path, index=False)

    rows = []
    for cl, sub in a.groupby("cluster"):
        ids = sub["bodyId"].astype(int).tolist()
        rows.append({
            "dataset": dataset_key,
            "dataset_label": ds["label"],
            "cluster": int(cl),
            "cluster_name": f"{dataset_key}_C{int(cl):02d}",
            "n_cells": len(ids),
            "bodyIds_comma": ",".join(map(str, ids)),
            "bodyIds_semicolon": ";".join(map(str, ids)),
        })
    cluster_df = pd.DataFrame(rows).sort_values(["n_cells", "cluster"], ascending=[False, True])
    cluster_path = EXPORT_DIR / f"{dataset_key}_cluster_id_lists_loaded.csv"
    cluster_df.to_csv(cluster_path, index=False)

    print(f"Saved: {per_neuron_path}")
    print(f"Saved: {cluster_path}")

    if HAS_OPENPYXL:
        xlsx = EXPORT_DIR / f"{dataset_key}_cluster_ids_loaded.xlsx"
        with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
            a.to_excel(writer, sheet_name="per_neuron", index=False)
            cluster_df.to_excel(writer, sheet_name="per_cluster", index=False)
        print(f"Saved: {xlsx}")

    return cluster_df

all_cluster_summaries = []
for key, ds in datasets.items():
    all_cluster_summaries.append(export_loaded_cluster_ids(ds, key))

combined_cluster_summary = pd.concat(all_cluster_summaries, ignore_index=True)
combined_path = EXPORT_DIR / "combined_cluster_id_lists_loaded.csv"
combined_cluster_summary.to_csv(combined_path, index=False)
print(f"Saved: {combined_path}")
display(combined_cluster_summary)


In [ ]:
# ============================================================
# 5. SWC parser + display-only flipped coordinate transform
# ============================================================

SWC_COLS = ["node_id", "type", "x", "y", "z", "radius", "parent"]


def body_id_from_path(path):
    m = re.search(r"(\d+)", Path(path).stem)
    return int(m.group(1)) if m else None


def read_swc(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 7:
                continue
            try:
                rows.append([
                    int(float(parts[0])), int(float(parts[1])),
                    float(parts[2]), float(parts[3]), float(parts[4]),
                    float(parts[5]), int(float(parts[6])),
                ])
            except Exception:
                pass
    return pd.DataFrame(rows, columns=SWC_COLS)


def swc_edges(df):
    if df is None or df.empty:
        return []
    ids = set(df["node_id"].astype(int))
    edges = []
    for _, r in df.iterrows():
        p = int(r["parent"])
        n = int(r["node_id"])
        if p in ids and p != -1:
            edges.append((p, n))
    return edges


def estimate_soma_from_swc(df):
    if df is None or df.empty:
        return None
    # Prefer SWC type 1 soma node if available.
    soma_nodes = df.loc[df["type"].astype(int) == 1]
    if len(soma_nodes):
        r = soma_nodes.iloc[0]
        return np.array([r["x"], r["y"], r["z"]], dtype=float)
    # Then root.
    root = df.loc[(df["parent"].astype(int) == -1)]
    if len(root):
        r = root.iloc[0]
        return np.array([r["x"], r["y"], r["z"]], dtype=float)
    # Then largest-radius node.
    if "radius" in df.columns and df["radius"].notna().any():
        r = df.loc[df["radius"].astype(float).idxmax()]
        return np.array([r["x"], r["y"], r["z"]], dtype=float)
    return df[["x", "y", "z"]].median().values.astype(float)


def swc_path_for_body(body_id, swc_dir):
    swc_dir = Path(swc_dir)
    direct = swc_dir / f"{int(body_id)}.swc"
    if direct.exists():
        return direct
    matches = list(swc_dir.glob(f"*{int(body_id)}*.swc"))
    return matches[0] if matches else None


def swc_segments_for_body(body_id, swc_dir):
    path = swc_path_for_body(body_id, swc_dir)
    if path is None:
        return [], None
    df = read_swc(path)
    if df.empty:
        return [], None
    coords = df.set_index("node_id")[["x", "y", "z"]]
    segs = []
    for p, n in swc_edges(df):
        try:
            a = coords.loc[p].values.astype(float)
            b = coords.loc[n].values.astype(float)
            segs.append((a, b))
        except Exception:
            pass
    soma = estimate_soma_from_swc(df)
    return segs, soma

# Load the shell from the previous v14 run.
SHELL_NPZ = PREVIOUS_OUT_ROOT / "outer_brain_shell" / "outer_brain_shell_voxel_union.npz"
if SHELL_NPZ.exists():
    shell_data = np.load(SHELL_NPZ)
    outer_shell_vertices = shell_data["vertices"]
    outer_shell_faces = shell_data["faces"].astype(int)
    print("Loaded outer shell:", SHELL_NPZ, outer_shell_vertices.shape, outer_shell_faces.shape)
else:
    outer_shell_vertices = None
    outer_shell_faces = None
    print("No previous outer shell found:", SHELL_NPZ)


# ------------------------------------------------------------
# Clean brain shell mesh: remove disconnected ROI/mask blobs
# ------------------------------------------------------------
# The raw shell built from all neuPrint ROI meshes can include small disconnected
# components outside the brain. For figure-ready rendering we keep only the
# largest connected mesh component, which gives a cleaner translucent brain
# outline like the reference figure.
SHELL_KEEP_LARGEST_COMPONENT_ONLY = True
SHELL_MIN_COMPONENT_FACE_FRACTION = 0.02


def clean_outer_shell_mesh_largest_component(vertices, faces, min_face_fraction=SHELL_MIN_COMPONENT_FACE_FRACTION):
    """
    Keep only the largest connected component of a triangular mesh.

    Connectivity is defined by faces sharing vertices. This removes small
    disconnected ROI blobs / masks outside the main brain shell.
    """
    if vertices is None or faces is None:
        return vertices, faces, {"status": "missing"}

    vertices = np.asarray(vertices, dtype=float)
    faces = np.asarray(faces, dtype=int)

    if len(vertices) == 0 or len(faces) == 0:
        return vertices, faces, {"status": "empty"}

    from collections import defaultdict, deque

    vertex_to_faces = defaultdict(list)
    for fi, tri in enumerate(faces):
        for v in tri:
            vertex_to_faces[int(v)].append(fi)

    seen = np.zeros(len(faces), dtype=bool)
    components = []

    for start in range(len(faces)):
        if seen[start]:
            continue
        q = deque([start])
        seen[start] = True
        comp = []
        while q:
            fi = q.popleft()
            comp.append(fi)
            for v in faces[fi]:
                for nb in vertex_to_faces[int(v)]:
                    if not seen[nb]:
                        seen[nb] = True
                        q.append(nb)
        components.append(np.asarray(comp, dtype=int))

    components = sorted(components, key=len, reverse=True)
    largest = components[0]
    min_faces = max(1, int(len(faces) * float(min_face_fraction)))

    # Default: largest component only. This is what removes the little blobs.
    keep_faces = largest

    kept_faces = faces[keep_faces]
    used_vertices = np.unique(kept_faces.ravel())
    old_to_new = {int(old): i for i, old in enumerate(used_vertices)}
    remapped_faces = np.vectorize(old_to_new.get)(kept_faces).astype(int)
    cleaned_vertices = vertices[used_vertices]

    info = {
        "status": "cleaned",
        "n_components": len(components),
        "original_vertices": int(len(vertices)),
        "original_faces": int(len(faces)),
        "kept_vertices": int(len(cleaned_vertices)),
        "kept_faces": int(len(remapped_faces)),
        "largest_component_faces": int(len(largest)),
        "min_faces_threshold": int(min_faces),
        "removed_faces": int(len(faces) - len(remapped_faces)),
        "component_face_counts_top10": [int(len(c)) for c in components[:10]],
    }

    print("Brain shell cleaning:")
    print(f"  components found: {info['n_components']}")
    print(f"  original faces: {info['original_faces']:,}")
    print(f"  kept faces:     {info['kept_faces']:,}")
    print(f"  removed faces:  {info['removed_faces']:,}")
    print(f"  top component face counts: {info['component_face_counts_top10']}")

    return cleaned_vertices, remapped_faces, info


if SHELL_KEEP_LARGEST_COMPONENT_ONLY and outer_shell_vertices is not None and outer_shell_faces is not None:
    outer_shell_vertices, outer_shell_faces, SHELL_CLEANING_INFO = clean_outer_shell_mesh_largest_component(
        outer_shell_vertices,
        outer_shell_faces,
    )
else:
    SHELL_CLEANING_INFO = {"status": "not_cleaned"}


def compute_render_center():
    if outer_shell_vertices is not None and len(outer_shell_vertices):
        return np.nanmean(outer_shell_vertices, axis=0)
    pts = []
    for ds in datasets.values():
        swc_dir = ds.get("raw_swc_dir")
        for bid in ds.get("body_ids", [])[:50]:
            path = swc_path_for_body(bid, swc_dir)
            if path:
                df = read_swc(path)
                if len(df):
                    pts.append(df[["x", "y", "z"]].values)
    if pts:
        return np.nanmean(np.vstack(pts), axis=0)
    return np.array([0.0, 0.0, 0.0])

RENDER_CENTER = compute_render_center()
print("Render center:", RENDER_CENTER)


def transform_points_for_render(points):
    """Apply display-only reflection around RENDER_CENTER."""
    if points is None:
        return None
    arr = np.asarray(points, dtype=float).copy()
    was_1d = arr.ndim == 1
    if was_1d:
        arr = arr[None, :]
    if arr.shape[1] < 3:
        return points
    center = np.asarray(RENDER_CENTER, dtype=float)
    if RENDER_FLIP_X:
        arr[:, 0] = 2 * center[0] - arr[:, 0]
    if RENDER_FLIP_Y:
        arr[:, 1] = 2 * center[1] - arr[:, 1]
    if RENDER_FLIP_Z:
        arr[:, 2] = 2 * center[2] - arr[:, 2]
    return arr[0] if was_1d else arr

outer_shell_vertices_render = transform_points_for_render(outer_shell_vertices) if outer_shell_vertices is not None else None
print("Render flips applied: X=", RENDER_FLIP_X, "Y=", RENDER_FLIP_Y, "Z=", RENDER_FLIP_Z)


In [ ]:
# ============================================================
# 6. neuPrint soma metadata: real somaLocation/somaRadius when available
# ============================================================

def get_neuprint_token():
    token = os.environ.get("NEUPRINT_TOKEN", "")
    if token:
        return token
    token_file = PROJECT_DIR / "neuprint_token.txt"
    if token_file.exists():
        token = token_file.read_text().strip()
        if token:
            os.environ["NEUPRINT_TOKEN"] = token
            return token
    if NEUPRINT_TOKEN_FALLBACK:
        os.environ["NEUPRINT_TOKEN"] = NEUPRINT_TOKEN_FALLBACK
        return NEUPRINT_TOKEN_FALLBACK
    return ""


def make_neuprint_client_or_none():
    if not HAS_NEUPRINT:
        return None
    token = get_neuprint_token()
    if not token:
        print("No neuPrint token found. Soma metadata will fall back to SWC estimates.")
        return None
    try:
        return Client(NEUPRINT_SERVER, dataset=NEUPRINT_DATASET, token=token)
    except Exception as e:
        print("Could not create neuPrint client. Soma metadata will fall back to SWC estimates.")
        print(repr(e))
        return None

c = make_neuprint_client_or_none()


def parse_neuprint_location_value(v):
    if v is None:
        return None
    try:
        if isinstance(v, float) and np.isnan(v):
            return None
    except Exception:
        pass
    if isinstance(v, str):
        nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", v)
        if len(nums) >= 3:
            return np.array([float(nums[0]), float(nums[1]), float(nums[2])], dtype=float)
        return None
    if isinstance(v, (list, tuple, np.ndarray)) and len(v) >= 3:
        return np.array([float(v[0]), float(v[1]), float(v[2])], dtype=float)
    if isinstance(v, dict):
        if all(k in v for k in ["x", "y", "z"]):
            return np.array([float(v["x"]), float(v["y"]), float(v["z"])], dtype=float)
        if "coordinates" in v and len(v["coordinates"]) >= 3:
            vv = v["coordinates"]
            return np.array([float(vv[0]), float(vv[1]), float(vv[2])], dtype=float)
    if all(hasattr(v, k) for k in ["x", "y", "z"]):
        return np.array([float(v.x), float(v.y), float(v.z)], dtype=float)
    return None


def fetch_neuprint_soma_metadata(body_ids, label="all_datasets"):
    body_ids = sorted(set(int(x) for x in body_ids if pd.notna(x)))
    if not body_ids or c is None or not USE_NEUPRINT_SOMAS:
        return pd.DataFrame()
    out_csv = SOMA_DIR / f"{label}_soma_metadata.csv"
    if out_csv.exists() and not FORCE_REFETCH_SOMAS:
        df = pd.read_csv(out_csv)
        print(f"Loaded cached soma metadata: {out_csv} | rows={len(df)}")
        return df
    rows = []
    batch_size = 300
    for i in tqdm(range(0, len(body_ids), batch_size), desc=f"Fetch neuPrint soma metadata {label}"):
        batch = body_ids[i:i+batch_size]
        try:
            neurons, _ = fetch_neurons(NC(bodyId=batch), client=c)
        except Exception as e:
            print(f"Soma metadata fetch failed for batch {i//batch_size}: {repr(e)}")
            continue
        if neurons is None or neurons.empty:
            continue
        for _, r in neurons.iterrows():
            bid = int(r["bodyId"])
            loc = None
            source = None
            for col in SOMA_LOCATION_PRIORITY:
                if col in neurons.columns:
                    loc = parse_neuprint_location_value(r.get(col, None))
                    if loc is not None:
                        source = col
                        break
            radius = np.nan
            for rad_col in ["somaRadius", "radius"]:
                if rad_col in neurons.columns:
                    try:
                        radius = float(r.get(rad_col))
                    except Exception:
                        radius = np.nan
                    break
            rows.append({
                "bodyId": bid,
                "soma_x": np.nan if loc is None else loc[0],
                "soma_y": np.nan if loc is None else loc[1],
                "soma_z": np.nan if loc is None else loc[2],
                "soma_radius": radius,
                "soma_source": "neuprint_" + str(source) if source else "neuprint_missing",
            })
    df = pd.DataFrame(rows).drop_duplicates("bodyId")
    df.to_csv(out_csv, index=False)
    print(f"Saved soma metadata: {out_csv} | rows={len(df)}")
    return df

all_body_ids_for_somas = []
for ds in datasets.values():
    all_body_ids_for_somas.extend(ds["assignments"]["bodyId"].astype(int).tolist())

soma_metadata_df = fetch_neuprint_soma_metadata(all_body_ids_for_somas, label="all_loaded_datasets")

SOMA_LOOKUP = {}
if not soma_metadata_df.empty:
    for _, r in soma_metadata_df.iterrows():
        loc = np.array([r.get("soma_x", np.nan), r.get("soma_y", np.nan), r.get("soma_z", np.nan)], dtype=float)
        if np.isfinite(loc).all():
            SOMA_LOOKUP[int(r["bodyId"])] = {
                "xyz": loc,
                "radius": float(r.get("soma_radius", np.nan)) if pd.notna(r.get("soma_radius", np.nan)) else np.nan,
                "source": r.get("soma_source", "neuprint"),
            }
print(f"Soma lookup entries with finite coordinates: {len(SOMA_LOOKUP)}")


## Soma/cell-body rendering note

This plotting version separates the visual inspection into three views per cluster:

1. **Skeletons only** — old morphology skeleton traces without soma meshes.
2. **Somas only** — local SWC-radius soma/root meshes without projections.
3. **Combined** — skeleton traces + soma/root meshes.

This should make it much easier to judge whether the soma rendering helps or distracts, and whether projection patterns are consistent within a cluster. All cells in a cluster use the same stable cluster color across all three views.

If true per-body segmentation meshes are available locally, put files in `body_meshes/<bodyId>.obj` or `body_meshes/<bodyId>.ply` and use `SOMA_RENDER_MODE = "local_body_mesh"`.

In [ ]:
# ============================================================
# 7. 3D rendering helpers: shell + neurons + SWC-radius soma meshes
# ============================================================

if not HAS_PLOTLY:
    raise ImportError("Plotly is required for the interactive 3D cluster renderings.")


def rgba(hex_color, alpha=1.0):
    """Convert '#RRGGBB' to plotly rgba string."""
    hex_color = str(hex_color).lstrip("#")
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def add_shell_trace(fig, alpha=SHELL_ALPHA):
    if not SHOW_BRAIN_SHELL:
        return fig
    if outer_shell_vertices_render is None or outer_shell_faces is None or len(outer_shell_vertices_render) == 0:
        return fig
    v = outer_shell_vertices_render
    f = outer_shell_faces
    fig.add_trace(go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        opacity=alpha,
        color="rgb(210, 215, 220)",
        name="outer brain shell",
        showscale=False,
        hoverinfo="skip",
        flatshading=False,
        lighting=dict(ambient=0.82, diffuse=0.35, specular=0.05, roughness=0.95),
    ))
    return fig


def soma_for_body(body_id, swc_dir):
    """Return soma center/radius/source using neuPrint metadata, then SWC fallback."""
    bid = int(body_id)
    if bid in SOMA_LOOKUP:
        d = SOMA_LOOKUP[bid]
        return d["xyz"], d.get("radius", np.nan), d.get("source", "neuprint")
    _, swc_soma = swc_segments_for_body(bid, swc_dir)
    if swc_soma is not None:
        return swc_soma, np.nan, "swc_fallback"
    return None, np.nan, "missing"


def sphere_mesh_vertices_faces(center, radius, theta_steps=9, phi_steps=7):
    """Return vertices/faces for a small sphere mesh."""
    center = np.asarray(center, dtype=float)
    radius = float(radius)
    theta = np.linspace(0, 2*np.pi, theta_steps, endpoint=False)
    phi = np.linspace(0, np.pi, phi_steps)
    verts = []
    for p in phi:
        for t in theta:
            verts.append([
                center[0] + radius * np.cos(t) * np.sin(p),
                center[1] + radius * np.sin(t) * np.sin(p),
                center[2] + radius * np.cos(p),
            ])
    faces = []
    for pi in range(phi_steps - 1):
        for ti in range(theta_steps):
            a = pi * theta_steps + ti
            b = pi * theta_steps + ((ti + 1) % theta_steps)
            c = (pi + 1) * theta_steps + ti
            d = (pi + 1) * theta_steps + ((ti + 1) % theta_steps)
            faces.append([a, c, b])
            faces.append([b, c, d])
    return np.asarray(verts, dtype=float), np.asarray(faces, dtype=int)


def cylinder_mesh_vertices_faces(a, b, radius, sides=9):
    """Return vertices/faces for a cylinder between two 3D points."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    axis = b - a
    length = np.linalg.norm(axis)
    if not np.isfinite(length) or length <= 1e-9:
        return np.empty((0, 3)), np.empty((0, 3), dtype=int)
    axis = axis / length
    # Choose an arbitrary perpendicular basis.
    tmp = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(tmp, axis)) > 0.9:
        tmp = np.array([0.0, 1.0, 0.0])
    u = np.cross(axis, tmp)
    u = u / np.linalg.norm(u)
    v = np.cross(axis, u)
    angles = np.linspace(0, 2*np.pi, sides, endpoint=False)
    ring_a, ring_b = [], []
    for t in angles:
        off = radius * (np.cos(t) * u + np.sin(t) * v)
        ring_a.append(a + off)
        ring_b.append(b + off)
    verts = np.vstack([ring_a, ring_b])
    faces = []
    for i in range(sides):
        j = (i + 1) % sides
        faces.append([i, j, sides + i])
        faces.append([j, sides + j, sides + i])
    return verts, np.asarray(faces, dtype=int)


def combine_mesh_parts(parts):
    """Combine list of (vertices, faces) mesh parts."""
    verts_all = []
    faces_all = []
    offset = 0
    for verts, faces in parts:
        if verts is None or faces is None or len(verts) == 0 or len(faces) == 0:
            continue
        verts_all.append(verts)
        faces_all.append(faces + offset)
        offset += len(verts)
    if not verts_all:
        return np.empty((0, 3)), np.empty((0, 3), dtype=int)
    return np.vstack(verts_all), np.vstack(faces_all)


def add_mesh3d(fig, vertices, faces, color, name, opacity=0.9, hoverinfo="skip", showlegend=False):
    if vertices is None or faces is None or len(vertices) == 0 or len(faces) == 0:
        return fig
    v = transform_points_for_render(vertices)
    fig.add_trace(go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=color,
        opacity=opacity,
        name=name,
        showscale=False,
        hoverinfo=hoverinfo,
        flatshading=False,
        lighting=dict(ambient=0.45, diffuse=0.65, specular=0.2, roughness=0.75),
        showlegend=showlegend,
    ))
    return fig


def closest_swc_node_to_point(df, point):
    if df is None or df.empty or point is None:
        return None
    pts = df[["x", "y", "z"]].values.astype(float)
    point = np.asarray(point, dtype=float)
    d = np.linalg.norm(pts - point[None, :], axis=1)
    return int(df.iloc[int(np.nanargmin(d))]["node_id"])


def soma_anchor_node(df, soma_xyz=None):
    """Best node to anchor soma rendering."""
    if df is None or df.empty:
        return None
    if soma_xyz is not None and np.isfinite(np.asarray(soma_xyz, dtype=float)).all():
        n = closest_swc_node_to_point(df, soma_xyz)
        if n is not None:
            return n
    soma_nodes = df.loc[df["type"].astype(int) == 1]
    if len(soma_nodes):
        return int(soma_nodes.iloc[0]["node_id"])
    roots = df.loc[df["parent"].astype(int) == -1]
    if len(roots):
        return int(roots.iloc[0]["node_id"])
    if "radius" in df.columns and df["radius"].notna().any():
        return int(df.loc[df["radius"].astype(float).idxmax()]["node_id"])
    return int(df.iloc[0]["node_id"])


def local_soma_node_set(df, soma_xyz, anchor):
    """Choose compact local SWC node set around soma/root, emphasizing large radii."""
    df = df.copy()
    ids = set(df["node_id"].astype(int))
    coords = df.set_index("node_id")[["x", "y", "z"]]
    radii = df.set_index("node_id")["radius"].astype(float)
    adjacency = defaultdict(list)
    for p, n in swc_edges(df):
        if p in ids and n in ids:
            adjacency[int(p)].append(int(n))
            adjacency[int(n)].append(int(p))

    visited = {int(anchor): 0}
    frontier = [int(anchor)]
    for _ in range(SOMA_RADIUS_MESH_GRAPH_HOPS):
        new_frontier = []
        for u in frontier:
            for v in adjacency.get(u, []):
                if v not in visited:
                    visited[v] = visited[u] + 1
                    new_frontier.append(v)
        frontier = new_frontier
        if not frontier:
            break
    graph_nodes = set(visited.keys())

    if soma_xyz is None or not np.isfinite(np.asarray(soma_xyz, dtype=float)).all():
        soma_xyz = coords.loc[anchor].values.astype(float)
    soma_xyz = np.asarray(soma_xyz, dtype=float)
    all_pts = coords.values.astype(float)
    all_ids = coords.index.astype(int).to_numpy()
    d = np.linalg.norm(all_pts - soma_xyz[None, :], axis=1)
    euclidean_nodes = set(all_ids[d <= SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS].astype(int))

    # Include local high-radius nodes; this tends to capture soma/root swelling better than distance alone.
    rr = radii.reindex(all_ids).values.astype(float)
    valid_rr = rr[np.isfinite(rr) & (rr > 0)]
    high_radius_nodes = set()
    if len(valid_rr):
        threshold = np.nanpercentile(valid_rr, 85)
        high_radius_nodes = set(all_ids[(rr >= threshold) & (d <= SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS * 1.4)].astype(int))

    chosen = list((graph_nodes | euclidean_nodes | high_radius_nodes).intersection(ids))
    if len(chosen) < SOMA_RADIUS_MESH_MIN_NODES:
        nearest = all_ids[np.argsort(d)[:SOMA_RADIUS_MESH_MIN_NODES]].astype(int).tolist()
        chosen = list(set(chosen) | set(nearest))
    if len(chosen) > SOMA_RADIUS_MESH_MAX_NODES:
        chosen_arr = np.array(chosen, dtype=int)
        pts_chosen = coords.loc[chosen_arr].values.astype(float)
        dd = np.linalg.norm(pts_chosen - soma_xyz[None, :], axis=1)
        # Prefer nearby + high radius.
        rr_chosen = radii.loc[chosen_arr].values.astype(float)
        rr_norm = np.nan_to_num(rr_chosen / (np.nanmax(rr_chosen) if np.nanmax(rr_chosen) > 0 else 1), nan=0)
        score = dd - 0.5 * SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS * rr_norm
        chosen = chosen_arr[np.argsort(score)[:SOMA_RADIUS_MESH_MAX_NODES]].astype(int).tolist()
    return sorted(set(map(int, chosen)))


def build_swc_radius_soma_mesh(body_id, swc_dir):
    """Build a compact soma/root mesh from local SWC nodes/radii."""
    path = swc_path_for_body(body_id, swc_dir)
    if path is None:
        return None
    df = read_swc(path)
    if df.empty:
        return None

    soma_xyz, soma_radius, soma_source = soma_for_body(body_id, swc_dir)
    anchor = soma_anchor_node(df, soma_xyz)
    if anchor is None:
        return None

    coords = df.set_index("node_id")[["x", "y", "z"]]
    radii = df.set_index("node_id")["radius"].astype(float)
    chosen = local_soma_node_set(df, soma_xyz, anchor)
    chosen_set = set(chosen)

    # Radii: use SWC radius, with conservative clipping to avoid ugly giant blobs.
    rr = radii.loc[chosen].values.astype(float)
    good = np.isfinite(rr) & (rr > 0)
    fallback_r = np.nanmedian(rr[good]) if np.any(good) else (soma_radius if np.isfinite(soma_radius) and soma_radius > 0 else 60.0)
    rr = np.where(good, rr, fallback_r)
    rr = np.clip(rr * SOMA_RADIUS_MESH_RADIUS_SCALE, SOMA_RADIUS_MESH_MIN_RADIUS, SOMA_RADIUS_MESH_MAX_RADIUS)

    parts = []
    # Sphere caps at selected local nodes.
    for nid, radius in zip(chosen, rr):
        center = coords.loc[nid].values.astype(float)
        parts.append(sphere_mesh_vertices_faces(
            center, radius,
            theta_steps=SOMA_RADIUS_MESH_SPHERE_THETA,
            phi_steps=SOMA_RADIUS_MESH_SPHERE_PHI,
        ))

    # Cylinders along local SWC edges. Radius is the smaller of endpoint radii.
    rr_by_id = dict(zip(chosen, rr))
    for p, n in swc_edges(df):
        if p in chosen_set and n in chosen_set:
            a = coords.loc[p].values.astype(float)
            b = coords.loc[n].values.astype(float)
            rad = min(rr_by_id.get(p, fallback_r), rr_by_id.get(n, fallback_r))
            parts.append(cylinder_mesh_vertices_faces(a, b, rad, sides=SOMA_RADIUS_MESH_CYLINDER_SIDES))

    vertices, faces = combine_mesh_parts(parts)
    if len(vertices) == 0 or len(faces) == 0:
        return None
    return {
        "vertices": vertices,
        "faces": faces,
        "source": soma_source,
        "n_nodes": len(chosen),
        "center": soma_xyz if soma_xyz is not None else coords.loc[anchor].values.astype(float),
    }


def read_obj_mesh(path):
    verts, faces = [], []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line.startswith("v "):
                parts = line.split()
                if len(parts) >= 4:
                    verts.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif line.startswith("f "):
                inds = []
                for p in line.split()[1:]:
                    try:
                        inds.append(int(p.split("/")[0]) - 1)
                    except Exception:
                        pass
                if len(inds) >= 3:
                    # Triangulate fan if needed.
                    for i in range(1, len(inds)-1):
                        faces.append([inds[0], inds[i], inds[i+1]])
    return np.asarray(verts, dtype=float), np.asarray(faces, dtype=int)


def read_ply_ascii_mesh(path):
    """Minimal ASCII PLY reader. Binary PLY is not supported here."""
    with open(path, "r", errors="ignore") as f:
        lines = f.readlines()
    if not lines or not lines[0].startswith("ply"):
        raise ValueError("Not a PLY file")
    n_vertices = n_faces = None
    end_idx = None
    for i, line in enumerate(lines):
        if line.startswith("format") and "ascii" not in line:
            raise ValueError("Only ASCII PLY supported by this lightweight reader")
        if line.startswith("element vertex"):
            n_vertices = int(line.split()[-1])
        if line.startswith("element face"):
            n_faces = int(line.split()[-1])
        if line.strip() == "end_header":
            end_idx = i + 1
            break
    if end_idx is None or n_vertices is None:
        raise ValueError("Malformed PLY")
    verts = []
    for line in lines[end_idx:end_idx+n_vertices]:
        parts = line.split()
        verts.append([float(parts[0]), float(parts[1]), float(parts[2])])
    faces = []
    face_start = end_idx + n_vertices
    for line in lines[face_start:face_start+(n_faces or 0)]:
        vals = [int(x) for x in line.split()]
        k = vals[0]
        inds = vals[1:1+k]
        for i in range(1, len(inds)-1):
            faces.append([inds[0], inds[i], inds[i+1]])
    return np.asarray(verts, dtype=float), np.asarray(faces, dtype=int)


def local_body_mesh_path(body_id):
    if not LOCAL_BODY_MESH_DIR.exists():
        return None
    for ext in [".obj", ".ply"]:
        p = LOCAL_BODY_MESH_DIR / f"{int(body_id)}{ext}"
        if p.exists():
            return p
    matches = []
    for ext in ["*.obj", "*.ply"]:
        matches.extend(LOCAL_BODY_MESH_DIR.glob(f"*{int(body_id)}*{ext[-4:]}"))
    return matches[0] if matches else None


def clipped_local_body_soma_mesh(body_id, swc_dir):
    """If local true body mesh exists, clip soma neighborhood and return mesh."""
    path = local_body_mesh_path(body_id)
    if path is None:
        return None
    soma_xyz, soma_radius, source = soma_for_body(body_id, swc_dir)
    if soma_xyz is None:
        return None
    try:
        if str(path).lower().endswith(".obj"):
            verts, faces = read_obj_mesh(path)
        else:
            verts, faces = read_ply_ascii_mesh(path)
    except Exception as e:
        print(f"Could not read local body mesh for {body_id}: {e}")
        return None
    if len(verts) == 0 or len(faces) == 0:
        return None
    d = np.linalg.norm(verts - np.asarray(soma_xyz)[None, :], axis=1)
    keep_v = d <= LOCAL_BODY_MESH_SOMA_CLIP_RADIUS
    keep_faces = keep_v[faces].all(axis=1)
    faces2 = faces[keep_faces]
    if len(faces2) == 0:
        return None
    # Limit if too large.
    if len(faces2) > LOCAL_BODY_MESH_MAX_FACES:
        rng = np.random.default_rng(42)
        faces2 = faces2[np.sort(rng.choice(len(faces2), LOCAL_BODY_MESH_MAX_FACES, replace=False))]
    used = np.unique(faces2.ravel())
    remap = {old: new for new, old in enumerate(used)}
    verts_new = verts[used]
    faces_new = np.vectorize(remap.get)(faces2)
    return {"vertices": verts_new, "faces": faces_new.astype(int), "source": f"local_body_mesh:{path.name}"}


def add_soma_trace(fig, body_id, swc_dir, color, show_text=False):
    if SOMA_RENDER_MODE == "local_body_mesh" or (PREFER_LOCAL_BODY_MESH_SOMA and SOMA_RENDER_MODE == "swc_radius_mesh"):
        mesh = clipped_local_body_soma_mesh(body_id, swc_dir)
        if mesh is not None:
            fig = add_mesh3d(
                fig, mesh["vertices"], mesh["faces"],
                color=color,
                name=f"true/local soma mesh {int(body_id)}",
                opacity=LOCAL_BODY_MESH_OPACITY,
                hoverinfo="text",
                showlegend=False,
            )
            return fig, True

    if SOMA_RENDER_MODE == "swc_radius_mesh":
        mesh = build_swc_radius_soma_mesh(body_id, swc_dir)
        if mesh is not None:
            fig = add_mesh3d(
                fig, mesh["vertices"], mesh["faces"],
                color=color,
                name=f"SWC-radius soma mesh {int(body_id)}",
                opacity=SOMA_RADIUS_MESH_OPACITY,
                hoverinfo="skip",
                showlegend=False,
            )
            # Optional label at mesh center.
            if show_text:
                c = transform_points_for_render(mesh["center"])
                fig.add_trace(go.Scatter3d(
                    x=[c[0]], y=[c[1]], z=[c[2]], mode="text",
                    text=[str(int(body_id))], textposition="top center",
                    showlegend=False, hoverinfo="skip",
                ))
            return fig, str(mesh.get("source", "")).startswith("neuprint")

    # Simple marker/sphere fallback.
    soma, radius, source = soma_for_body(body_id, swc_dir)
    if soma is None:
        return fig, False
    soma_r = transform_points_for_render(soma)
    label = f"soma {int(body_id)} ({source})"
    if SOMA_RENDER_MODE == "sphere" and np.isfinite(radius) and radius > 0:
        verts, faces = sphere_mesh_vertices_faces(soma, radius * SOMA_SPHERE_RADIUS_SCALE, theta_steps=10, phi_steps=8)
        fig = add_mesh3d(fig, verts, faces, color=color, name=label, opacity=SOMA_SPHERE_OPACITY)
    else:
        size = SOMA_MARKER_SIZE
        if np.isfinite(radius) and radius > 0:
            size = int(np.clip(radius / 50, SOMA_MARKER_SIZE_MIN, SOMA_MARKER_SIZE_MAX))
        fig.add_trace(go.Scatter3d(
            x=[soma_r[0]], y=[soma_r[1]], z=[soma_r[2]],
            mode="markers+text" if show_text else "markers",
            marker=dict(size=size, symbol="circle", color=color),
            text=[str(int(body_id))] if show_text else None,
            textposition="top center",
            name=label,
            hovertext=label,
            hoverinfo="text",
            showlegend=False,
        ))
    return fig, source.startswith("neuprint")


def build_neuron_radius_tube_mesh(body_id, swc_dir):
    """Experimental full-skeleton tube mesh from SWC radii. Heavy but tunable."""
    path = swc_path_for_body(body_id, swc_dir)
    if path is None:
        return None
    df = read_swc(path)
    if df.empty:
        return None
    coords = df.set_index("node_id")[["x", "y", "z"]]
    radii = df.set_index("node_id")["radius"].astype(float)
    parts = []
    edges = swc_edges(df)[:CELL_TUBE_MAX_EDGES_PER_NEURON]
    for p, n in edges:
        try:
            a = coords.loc[p].values.astype(float)
            b = coords.loc[n].values.astype(float)
            r = np.nanmean([radii.loc[p], radii.loc[n]])
            if not np.isfinite(r) or r <= 0:
                r = CELL_TUBE_MIN_RADIUS
            r = np.clip(r * CELL_TUBE_RADIUS_SCALE, CELL_TUBE_MIN_RADIUS, CELL_TUBE_MAX_RADIUS)
            parts.append(cylinder_mesh_vertices_faces(a, b, r, sides=CELL_TUBE_SIDES))
        except Exception:
            pass
    verts, faces = combine_mesh_parts(parts)
    if len(verts) == 0:
        return None
    return {"vertices": verts, "faces": faces}


def add_single_neuron_trace(fig, body_id, swc_dir, cluster_color, width=CELL_LINE_WIDTH):
    if CELL_RENDER_MODE == "radius_tubes":
        mesh = build_neuron_radius_tube_mesh(body_id, swc_dir)
        if mesh is not None:
            return add_mesh3d(
                fig, mesh["vertices"], mesh["faces"],
                color=rgba(cluster_color, CELL_LINE_ALPHA),
                name=f"neuron radius tube {int(body_id)}",
                opacity=CELL_LINE_ALPHA,
                hoverinfo="skip",
                showlegend=False,
            )
    # Fast clean line rendering fallback/default.
    segs, _ = swc_segments_for_body(body_id, swc_dir)
    if not segs:
        return fig
    xs, ys, zs = [], [], []
    for a, b in segs:
        aa = transform_points_for_render(a)
        bb = transform_points_for_render(b)
        xs += [aa[0], bb[0], None]
        ys += [aa[1], bb[1], None]
        zs += [aa[2], bb[2], None]
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(width=width, color=rgba(cluster_color, CELL_LINE_ALPHA)),
        name=f"cluster neuron {int(body_id)}",
        hovertext=str(int(body_id)),
        hoverinfo="text",
        showlegend=False,
    ))
    return fig


def add_neuron_traces(fig, body_ids, swc_dir, cluster_color, width=CELL_LINE_WIDTH, show_somas=True, show_soma_text=False):
    n_soma_neuprint = 0
    for bid in body_ids:
        fig = add_single_neuron_trace(fig, bid, swc_dir, cluster_color=cluster_color, width=width)
        if show_somas:
            fig, is_neuprint = add_soma_trace(fig, bid, swc_dir, color=cluster_color, show_text=show_soma_text)
            n_soma_neuprint += int(is_neuprint)
    return fig, n_soma_neuprint


def add_population_context(fig, ds, max_cells=MAX_CONTEXT_CELLS):
    a = ds["assignments"]
    ids = a["bodyId"].astype(int).tolist()[:max_cells]
    swc_dir = ds["raw_swc_dir"]
    for bid in ids:
        segs, _ = swc_segments_for_body(bid, swc_dir)
        if not segs:
            continue
        xs, ys, zs = [], [], []
        for p, q in segs:
            pp = transform_points_for_render(p)
            qq = transform_points_for_render(q)
            xs += [pp[0], qq[0], None]
            ys += [pp[1], qq[1], None]
            zs += [pp[2], qq[2], None]
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="lines",
            line=dict(width=CONTEXT_LINE_WIDTH, color=f"rgba(0,0,0,{POPULATION_CONTEXT_ALPHA})"),
            showlegend=False,
            hoverinfo="skip",
        ))
    return fig


def render_cluster_3d(ds, dataset_key, cluster, save_html=True, show_population_context=False, show_somas=True, show_soma_text=False):
    a = ds["assignments"]
    cluster = int(cluster)
    body_ids = a.loc[a["cluster"] == cluster, "bodyId"].astype(int).tolist()
    swc_dir = ds["raw_swc_dir"]
    ccolor = get_cluster_color(dataset_key, cluster)

    fig = go.Figure()
    add_shell_trace(fig, alpha=SHELL_ALPHA)
    if show_population_context:
        add_population_context(fig, ds)
    body_ids_to_draw = body_ids[:MAX_CELLS_PER_CLUSTER_RENDER]
    fig, n_neuprint_somas = add_neuron_traces(
        fig,
        body_ids_to_draw,
        swc_dir,
        cluster_color=ccolor,
        width=CELL_LINE_WIDTH,
        show_somas=show_somas,
        show_soma_text=show_soma_text,
    )

    id_text = ", ".join(map(str, body_ids[:35])) + ("..." if len(body_ids) > 35 else "")
    title = (
        f"{ds['label']} | cluster {cluster} | n={len(body_ids)}"
        f"<br>cluster color: {ccolor} | skeleton mode: {CELL_RENDER_MODE} | line width: {CELL_LINE_WIDTH} | soma mode: {SOMA_RENDER_MODE}"
        f"<br>neuPrint soma centers used: {n_neuprint_somas}/{len(body_ids_to_draw)} | IDs: {id_text}"
    )
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z",
            aspectmode="data",
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False),
        ),
        width=1050,
        height=850,
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=110, b=0),
    )
    if save_html and SAVE_INTERACTIVE_HTML:
        outdir = RENDER_DIR / dataset_key / "interactive_3d_v17_radius_soma_meshes"
        outdir.mkdir(parents=True, exist_ok=True)
        html = outdir / f"{dataset_key}_cluster_{cluster:02d}_v17_radiusSomaMesh.html"
        fig.write_html(str(html))
        print(f"Saved HTML: {html}")
    return fig


def inspect_cluster(dataset_key="selected_raphe", cluster=1, show_population_context=False, show_somas=True, show_soma_text=False):
    ds = datasets[dataset_key]
    ids = ds["assignments"].loc[ds["assignments"]["cluster"] == int(cluster), "bodyId"].astype(int).tolist()
    print(f"{ds['label']} | cluster {int(cluster)} | n={len(ids)} | color={get_cluster_color(dataset_key, cluster)}")
    print("IDs:", ", ".join(map(str, ids)))
    display(pd.DataFrame({"bodyId": ids}))
    fig = render_cluster_3d(
        ds, dataset_key, int(cluster),
        save_html=True,
        show_population_context=show_population_context,
        show_somas=show_somas,
        show_soma_text=show_soma_text,
    )
    display(fig)
    return fig


# ============================================================
# v18 overrides: render skeleton-only, soma-only, or combined views
# ============================================================

RENDER_MODE_LABELS = {
    "skeletons_only": "Skeletons only",
    "somas_only": "Somas only",
    "combined": "Skeletons + somas",
}


def _safe_cluster_ids(ds, cluster):
    a = ds["assignments"]
    cluster = int(cluster)
    return a.loc[a["cluster"] == cluster, "bodyId"].astype(int).tolist()


def render_cluster_3d_mode(
    ds,
    dataset_key,
    cluster,
    render_mode="combined",
    save_html=True,
    show_population_context=False,
    show_soma_text=False,
):
    """
    Render one cluster in one of three modes:
      - skeletons_only: only the old skeleton traces
      - somas_only: only soma/root radius meshes
      - combined: skeleton traces + soma/root meshes
    """
    if render_mode not in RENDER_MODE_LABELS:
        raise ValueError(f"render_mode must be one of {list(RENDER_MODE_LABELS)}")

    cluster = int(cluster)
    body_ids = _safe_cluster_ids(ds, cluster)
    body_ids_to_draw = body_ids[:MAX_CELLS_PER_CLUSTER_RENDER]
    swc_dir = ds["raw_swc_dir"]
    ccolor = get_cluster_color(dataset_key, cluster)

    fig = go.Figure()
    add_shell_trace(fig, alpha=SHELL_ALPHA)

    # Context is usually most useful in skeleton/combined mode; allow it everywhere if requested.
    if show_population_context:
        add_population_context(fig, ds)

    n_somas = 0
    for bid in body_ids_to_draw:
        if render_mode in ["skeletons_only", "combined"]:
            fig = add_single_neuron_trace(
                fig,
                bid,
                swc_dir,
                cluster_color=ccolor,
                width=CELL_LINE_WIDTH,
            )
        if render_mode in ["somas_only", "combined"]:
            fig, is_neuprint = add_soma_trace(
                fig,
                bid,
                swc_dir,
                color=ccolor,
                show_text=show_soma_text,
            )
            n_somas += int(is_neuprint)

    id_text = ", ".join(map(str, body_ids[:35])) + ("..." if len(body_ids) > 35 else "")
    title = (
        f"{ds['label']} | cluster {cluster} | n={len(body_ids)} | {RENDER_MODE_LABELS[render_mode]}"
        f"<br>cluster color: {ccolor} | line width: {CELL_LINE_WIDTH} | soma mode: {SOMA_RENDER_MODE}"
        f"<br>somas drawn: {len(body_ids_to_draw) if render_mode in ['somas_only', 'combined'] else 0}; "
        f"neuPrint soma anchors: {n_somas}/{len(body_ids_to_draw)} | IDs: {id_text}"
    )

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z",
            aspectmode="data",
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False),
        ),
        width=1050,
        height=850,
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=120, b=0),
    )

    if save_html and SAVE_INTERACTIVE_HTML and SAVE_EACH_RENDER_MODE_HTML:
        outdir = RENDER_DIR / dataset_key / "interactive_3d_v18_three_brain_views"
        outdir.mkdir(parents=True, exist_ok=True)
        html = outdir / f"{dataset_key}_cluster_{cluster:02d}_{render_mode}_v18.html"
        fig.write_html(str(html))
        print(f"Saved HTML: {html}")

    return fig


def render_cluster_three_views(
    ds,
    dataset_key,
    cluster,
    save_html=True,
    show_population_context=False,
    show_soma_text=False,
):
    """Render the three requested views as three separate interactive Plotly brains."""
    figs = {}
    for mode in ["skeletons_only", "somas_only", "combined"]:
        print(f"Rendering {RENDER_MODE_LABELS[mode]}...")
        figs[mode] = render_cluster_3d_mode(
            ds,
            dataset_key,
            cluster,
            render_mode=mode,
            save_html=save_html,
            show_population_context=show_population_context,
            show_soma_text=show_soma_text,
        )
    return figs


# Backward-compatible wrapper. Existing calls still work, but default to combined.
def render_cluster_3d(
    ds,
    dataset_key,
    cluster,
    save_html=True,
    show_population_context=False,
    show_somas=True,
    show_soma_text=False,
    render_mode="combined",
):
    if not show_somas and render_mode == "combined":
        render_mode = "skeletons_only"
    return render_cluster_3d_mode(
        ds,
        dataset_key,
        cluster,
        render_mode=render_mode,
        save_html=save_html,
        show_population_context=show_population_context,
        show_soma_text=show_soma_text,
    )


def inspect_cluster(
    dataset_key="selected_raphe",
    cluster=1,
    show_population_context=False,
    show_somas=True,
    show_soma_text=False,
    render_mode=DEFAULT_CLUSTER_RENDER_MODE,
):
    """
    Inspect a cluster. By default, render_mode='all_three' shows:
    skeletons-only, somas-only, and combined.
    """
    ds = datasets[dataset_key]
    ids = _safe_cluster_ids(ds, cluster)
    print(f"{ds['label']} | cluster {int(cluster)} | n={len(ids)} | color={get_cluster_color(dataset_key, cluster)}")
    print("IDs:", ", ".join(map(str, ids)))
    display(pd.DataFrame({"bodyId": ids}))

    if render_mode == "all_three":
        figs = render_cluster_three_views(
            ds,
            dataset_key,
            int(cluster),
            save_html=True,
            show_population_context=show_population_context,
            show_soma_text=show_soma_text,
        )
        for mode, fig in figs.items():
            display(HTML(f"<h3>{RENDER_MODE_LABELS[mode]}</h3>"))
            display(fig)
        return figs

    fig = render_cluster_3d_mode(
        ds,
        dataset_key,
        int(cluster),
        render_mode=render_mode,
        save_html=True,
        show_population_context=show_population_context,
        show_soma_text=show_soma_text,
    )
    display(fig)
    return fig


In [ ]:
# ============================================================
# 8. Interactive cluster selector: three separate brain views
# ============================================================

def launch_cluster_selector():
    if not HAS_WIDGETS:
        print("ipywidgets not available. Use inspect_cluster('selected_raphe', 1, render_mode='all_three').")
        return

    dataset_options = [(ds["label"], key) for key, ds in datasets.items()]
    dataset_dd = widgets.Dropdown(options=dataset_options, description="Dataset:", layout=widgets.Layout(width="420px"))
    cluster_dd = widgets.Dropdown(description="Cluster:", layout=widgets.Layout(width="260px"))
    render_mode_dd = widgets.Dropdown(
        options=[
            ("All three: skeletons, somas, combined", "all_three"),
            ("Skeletons only", "skeletons_only"),
            ("Somas only", "somas_only"),
            ("Combined", "combined"),
        ],
        value=DEFAULT_CLUSTER_RENDER_MODE,
        description="View:",
        layout=widgets.Layout(width="420px"),
    )
    show_context = widgets.Checkbox(value=False, description="Population context")
    show_soma_text = widgets.Checkbox(value=False, description="Soma ID labels")
    out = widgets.Output()

    def update_clusters(*args):
        key = dataset_dd.value
        ds = datasets[key]
        sizes = ds["assignments"]["cluster"].value_counts().sort_index()
        opts = [(f"C{int(c):02d} | n={int(s)}", int(c)) for c, s in sizes.items()]
        cluster_dd.options = opts

    def draw(*args):
        with out:
            clear_output(wait=True)
            key = dataset_dd.value
            cl = cluster_dd.value
            if cl is None:
                return
            inspect_cluster(
                key,
                cl,
                show_population_context=show_context.value,
                show_soma_text=show_soma_text.value,
                render_mode=render_mode_dd.value,
            )

    dataset_dd.observe(update_clusters, names="value")
    cluster_dd.observe(draw, names="value")
    render_mode_dd.observe(draw, names="value")
    show_context.observe(draw, names="value")
    show_soma_text.observe(draw, names="value")

    update_clusters()
    display(widgets.VBox([
        widgets.HTML("<b>Interactive cluster renderer — choose skeletons-only, somas-only, combined, or all three</b>"),
        widgets.HBox([dataset_dd, cluster_dd]),
        widgets.HBox([render_mode_dd]),
        widgets.HBox([show_context, show_soma_text]),
        out,
    ]))
    draw()

launch_cluster_selector()


In [ ]:
# ============================================================
# 9. Optional: batch-generate HTML renders for all non-singleton clusters
# ============================================================

# Set to True and rerun this cell if you want to regenerate HTML for every non-singleton cluster.
BATCH_RENDER_ALL_NON_SINGLETON_CLUSTERS = False

# Choose which views to batch-save.
# Options: ["skeletons_only", "somas_only", "combined"]
BATCH_RENDER_MODES = ["skeletons_only", "somas_only", "combined"]

if BATCH_RENDER_ALL_NON_SINGLETON_CLUSTERS:
    for key, ds in datasets.items():
        sizes = ds["assignments"]["cluster"].value_counts().sort_values(ascending=False)
        clusters = [int(c) for c, s in sizes.items() if int(s) >= 2]
        print(f"{key}: rendering {len(clusters)} non-singleton clusters")
        for cl in tqdm(clusters, desc=f"Render {key}"):
            for mode in BATCH_RENDER_MODES:
                _ = render_cluster_3d_mode(
                    ds,
                    key,
                    cl,
                    render_mode=mode,
                    save_html=True,
                    show_population_context=False,
                    show_soma_text=False,
                )
else:
    print("Batch rendering disabled. Set BATCH_RENDER_ALL_NON_SINGLETON_CLUSTERS = True to save all cluster HTML renders.")


In [ ]:
# ============================================================
# 10. Check which cluster a list of cell IDs belongs to
# ============================================================

# Edit this list and rerun the cell.
CELLS_TO_CHECK = [
    # 100010951,
    # 100014398,
]


def check_cells_in_existing_clusters(cell_ids=None):
    """
    Search both loaded datasets and report which cluster each cell belongs to.
    This only checks already-clustered QC-passing cells from the previous v14 run.
    """
    if cell_ids is None:
        cell_ids = CELLS_TO_CHECK
    cell_ids = [int(x) for x in cell_ids]
    rows = []
    for bid in cell_ids:
        found = False
        for dataset_key, ds in datasets.items():
            a = ds["assignments"]
            hit = a.loc[a["bodyId"].astype(int) == int(bid)]
            if len(hit):
                for _, r in hit.iterrows():
                    cl = int(r["cluster"])
                    cluster_ids = a.loc[a["cluster"] == cl, "bodyId"].astype(int).tolist()
                    rows.append({
                        "bodyId": int(bid),
                        "dataset": dataset_key,
                        "dataset_label": ds["label"],
                        "cluster": cl,
                        "cluster_name": f"{dataset_key}_C{cl:02d}",
                        "cluster_size": len(cluster_ids),
                        "cluster_bodyIds_semicolon": ";".join(map(str, cluster_ids)),
                        "status": "clustered",
                    })
                found = True
        if not found:
            rows.append({
                "bodyId": int(bid),
                "dataset": "",
                "dataset_label": "",
                "cluster": np.nan,
                "cluster_name": "",
                "cluster_size": np.nan,
                "cluster_bodyIds_semicolon": "",
                "status": "not_found_in_loaded_QC_passing_clusters",
            })
    out = pd.DataFrame(rows)
    display(out)
    path = PLOT_OUT_ROOT / "checked_cell_cluster_membership_loaded_from_v14.csv"
    out.to_csv(path, index=False)
    print(f"Saved: {path}")
    return out

checked_cells = check_cells_in_existing_clusters()


## 11. Figure-ready exports — fixed 3D shell views

This section replaces the older 2D projection panels. The old panels used a projected/convex-hull brain outline, which looked ugly and did **not** match the nice interactive 3D brain.

The functions below reuse the same Plotly 3D rendering machinery as the interactive cluster viewer, including the same translucent outer brain shell mesh. The only difference is that the camera is fixed to top, side, and angled views and each view is exported as PDF/PNG/SVG.


In [ ]:

# ============================================================
# 11. Figure-ready export configuration — SOMA-IN-RAPHE-SUPERIOR
# ============================================================

FIGURE_READY_DIR = PLOT_OUT_ROOT / "figure_ready_v20_fixed_3d_shell_soma_in_roi"
FIGURE_READY_DIR.mkdir(parents=True, exist_ok=True)

FIG_UMAP_DIR = FIGURE_READY_DIR / "umap_mds"
FIG_CLUSTER_3D_DIR = FIGURE_READY_DIR / "cluster_3d_static_views"
FIG_SINGLE_CLUSTER_DIR = FIGURE_READY_DIR / "single_cluster_custom_3d"
for _p in [FIG_UMAP_DIR, FIG_CLUSTER_3D_DIR, FIG_SINGLE_CLUSTER_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

# This notebook/version is for the Raphe-superior soma-in-ROI dataset.
FIGURE_DATASET_KEY = "raphe_superior_soma_in_roi"

# 3D static camera views.
FIGURE_3D_VIEWS = ["top", "side", "angled"]
FIGURE_3D_VIEW_LABELS = {
    "top": "Top view",
    "side": "Side view",
    "angled": "Angled view",
}

# Render mode options: "skeletons_only", "somas_only", "combined".
FIGURE_3D_RENDER_MODE = "combined"

# Plotly static output size. Bigger width/height gives nicer PDF/PNG.
FIGURE_3D_WIDTH = 1450
FIGURE_3D_HEIGHT = 1150
FIGURE_3D_SCALE = 2

# These control the same interactive 3D renderer; tune for less crowded figures.
FIGURE_3D_SHELL_ALPHA = 0.060
FIGURE_3D_CELL_LINE_WIDTH = 2.6
FIGURE_3D_CONTEXT_LINE_WIDTH = 0.45
FIGURE_3D_SHOW_CONTEXT = False
FIGURE_3D_SHOW_SOMA_TEXT = False
FIGURE_3D_MAX_CELLS = 120

# Batch rendering controls.
FIGURE_TOP_N_CLUSTERS = 12
FIGURE_MIN_CLUSTER_SIZE_TO_RENDER = 2

# UMAP style.
FIGURE_UMAP_POINT_SIZE = 34
FIGURE_UMAP_ALPHA = 0.92
FIGURE_UMAP_EDGE_LW = 0.3
FIGURE_UMAP_LABEL_CLUSTERS = True
FIGURE_UMAP_LABEL_POINTS = False

# Custom cluster defaults.
CUSTOM_DATASET_KEY = FIGURE_DATASET_KEY
CUSTOM_CLUSTER_ID = 1
CUSTOM_3D_VIEWS = ["top", "side", "angled"]
CUSTOM_RENDER_MODE_3D = "combined"
CUSTOM_SHOW_CONTEXT = False
CUSTOM_SHOW_SOMA_TEXT_IN_3D = False
CUSTOM_CELL_LINE_WIDTH = 3.0
CUSTOM_CONTEXT_LINE_WIDTH = 0.45
CUSTOM_SHELL_ALPHA = 0.060
CUSTOM_MAX_CELLS = 120

print("Fixed 3D figure-ready outputs will be saved to:", FIGURE_READY_DIR)
print("Using dataset:", FIGURE_DATASET_KEY)
print("Available datasets:", list(datasets.keys()))


In [ ]:

# ============================================================
# 12. Figure-ready helper functions — UMAPs + fixed-camera 3D exports
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display


def _require_fixed_3d_figure_objects():
    required = [
        "datasets",
        "render_cluster_3d_mode",
        "cluster_body_ids",
        "get_cluster_color",
        "outer_shell_vertices_render",
        "outer_shell_faces",
    ]
    missing = [x for x in required if x not in globals()]
    if missing:
        raise RuntimeError(
            "Missing required variables/functions for fixed 3D exports: "
            + ", ".join(missing)
            + ". Run the earlier loading/rendering cells first."
        )
    if not HAS_PLOTLY:
        raise RuntimeError("Plotly is required for the fixed 3D figure exports.")


def save_matplotlib_all_formats(fig, out_base, dpi=450):
    out_base = Path(out_base)
    out_base.parent.mkdir(parents=True, exist_ok=True)
    paths = []
    if SAVE_PNG:
        p = out_base.with_suffix(".png")
        fig.savefig(p, dpi=dpi, bbox_inches="tight")
        paths.append(p)
    if SAVE_PDF:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        paths.append(p)
    if SAVE_SVG:
        p = out_base.with_suffix(".svg")
        fig.savefig(p, bbox_inches="tight")
        paths.append(p)
    for p in paths:
        print("Saved:", p)
    return paths


def save_plotly_all_formats(fig, out_base, scale=FIGURE_3D_SCALE, save_html=True):
    """Save a Plotly figure as HTML plus static PNG/PDF/SVG if enabled."""
    out_base = Path(out_base)
    out_base.parent.mkdir(parents=True, exist_ok=True)
    paths = []

    if save_html and SAVE_INTERACTIVE_HTML:
        p = out_base.with_suffix(".html")
        fig.write_html(str(p))
        paths.append(p)
        print("Saved HTML:", p)

    try:
        if SAVE_PNG:
            p = out_base.with_suffix(".png")
            fig.write_image(str(p), scale=scale)
            paths.append(p)
            print("Saved PNG:", p)
        if SAVE_PDF:
            p = out_base.with_suffix(".pdf")
            fig.write_image(str(p), scale=scale)
            paths.append(p)
            print("Saved PDF:", p)
        if SAVE_SVG:
            p = out_base.with_suffix(".svg")
            fig.write_image(str(p), scale=scale)
            paths.append(p)
            print("Saved SVG:", p)
    except Exception as e:
        print(
            "Static Plotly export failed. The HTML/displayed figure is still usable.\n"
            "Install/update kaleido in this environment with:\n"
            "  pip install -U kaleido\n"
            f"Error: {repr(e)}"
        )
    return paths


def fixed_3d_camera(view):
    """Camera presets for static Plotly 3D exports."""
    view = str(view).lower()
    if view == "top":
        return dict(eye=dict(x=0.0, y=0.0, z=2.35), up=dict(x=0, y=1, z=0))
    if view == "side":
        # True lateral view: look along y-axis, not x-axis.
        return dict(eye=dict(x=0.0, y=2.35, z=0.0), up=dict(x=0, y=0, z=1))
    if view == "front":
        return dict(eye=dict(x=0.0, y=2.35, z=0.0), up=dict(x=0, y=0, z=1))
    if view == "angled":
        return dict(eye=dict(x=1.55, y=1.65, z=1.15), up=dict(x=0, y=0, z=1))
    raise ValueError(f"Unknown fixed 3D view: {view}")


def polish_fixed_3d_figure(fig, title, view):
    """Make the same interactive 3D brain render look like a clean static figure."""
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center", font=dict(size=22)),
        scene=dict(
            xaxis=dict(visible=False, showbackground=False),
            yaxis=dict(visible=False, showbackground=False),
            zaxis=dict(visible=False, showbackground=False),
            aspectmode="data",
            camera=fixed_3d_camera(view),
            bgcolor="white",
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=0, r=0, t=70, b=0),
        width=FIGURE_3D_WIDTH,
        height=FIGURE_3D_HEIGHT,
        showlegend=False,
    )
    return fig


def _temporarily_set_render_globals(cell_line_width, context_line_width, shell_alpha, max_cells):
    """Context-manager-like helper for temporarily tuning the existing 3D renderer."""
    old = {}
    replacements = {
        "CELL_LINE_WIDTH": cell_line_width,
        "CONTEXT_LINE_WIDTH": context_line_width,
        "SHELL_ALPHA": shell_alpha,
        "MAX_CELLS_PER_CLUSTER_RENDER": max_cells,
    }
    for k, v in replacements.items():
        if k in globals() and v is not None:
            old[k] = globals()[k]
            globals()[k] = v
    return old


def _restore_render_globals(old):
    for k, v in old.items():
        globals()[k] = v


def make_fixed_3d_cluster_view(
    dataset_key=FIGURE_DATASET_KEY,
    cluster=1,
    view="angled",
    render_mode=FIGURE_3D_RENDER_MODE,
    show_context=FIGURE_3D_SHOW_CONTEXT,
    show_soma_text=FIGURE_3D_SHOW_SOMA_TEXT,
    cell_line_width=FIGURE_3D_CELL_LINE_WIDTH,
    context_line_width=FIGURE_3D_CONTEXT_LINE_WIDTH,
    shell_alpha=FIGURE_3D_SHELL_ALPHA,
    max_cells=FIGURE_3D_MAX_CELLS,
    out_dir=None,
    save_outputs=True,
    display_figure=True,
):
    """
    Make one fixed-camera 3D cluster figure using the SAME Plotly brain shell
    as the interactive renderer. This is the key fix relative to the old ugly
    2D shell projection panels.
    """
    _require_fixed_3d_figure_objects()
    if dataset_key not in datasets:
        raise ValueError(f"Unknown dataset_key={dataset_key!r}. Available: {list(datasets.keys())}")

    ds = datasets[dataset_key]
    cluster = int(cluster)
    ids = cluster_body_ids(ds, cluster)
    if len(ids) == 0:
        raise ValueError(f"No cells found for {dataset_key} cluster {cluster}")

    old = _temporarily_set_render_globals(cell_line_width, context_line_width, shell_alpha, max_cells)
    try:
        fig = render_cluster_3d_mode(
            ds,
            dataset_key,
            cluster,
            render_mode=render_mode,
            save_html=False,
            show_population_context=show_context,
            show_soma_text=show_soma_text,
        )
    finally:
        _restore_render_globals(old)

    id_preview = ", ".join(map(str, ids[:12])) + (", ..." if len(ids) > 12 else "")
    title = f"{ds['label']} | C{cluster:02d} | n={len(ids)} | {FIGURE_3D_VIEW_LABELS.get(view, view)}<br><sup>{id_preview}</sup>"
    fig = polish_fixed_3d_figure(fig, title=title, view=view)

    if display_figure:
        display(fig)

    if save_outputs:
        if out_dir is None:
            out_dir = FIG_CLUSTER_3D_DIR / dataset_key / f"C{cluster:02d}"
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_base = out_dir / f"{dataset_key}_C{cluster:02d}_{render_mode}_{view}_fixed3D"
        save_plotly_all_formats(fig, out_base, scale=FIGURE_3D_SCALE, save_html=True)

    return fig


def make_fixed_3d_cluster_views(
    dataset_key=FIGURE_DATASET_KEY,
    cluster=1,
    views=None,
    render_mode=FIGURE_3D_RENDER_MODE,
    show_context=FIGURE_3D_SHOW_CONTEXT,
    show_soma_text=FIGURE_3D_SHOW_SOMA_TEXT,
    cell_line_width=FIGURE_3D_CELL_LINE_WIDTH,
    context_line_width=FIGURE_3D_CONTEXT_LINE_WIDTH,
    shell_alpha=FIGURE_3D_SHELL_ALPHA,
    max_cells=FIGURE_3D_MAX_CELLS,
    out_dir=None,
    save_outputs=True,
    display_figures=True,
):
    """Make top/side/angled fixed-camera 3D figures and save each as PDF/PNG/SVG/HTML."""
    if views is None:
        views = FIGURE_3D_VIEWS

    ds = datasets[dataset_key]
    cluster = int(cluster)
    ids = cluster_body_ids(ds, cluster)

    if out_dir is None:
        out_dir = FIG_CLUSTER_3D_DIR / dataset_key / f"C{cluster:02d}"
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    id_table = pd.DataFrame({
        "dataset_key": dataset_key,
        "dataset_label": ds["label"],
        "cluster": cluster,
        "cluster_name": f"{dataset_key}_C{cluster:02d}",
        "n_cells": len(ids),
        "bodyId": ids,
    })
    id_csv = out_dir / f"{dataset_key}_C{cluster:02d}_cell_ids.csv"
    id_txt = out_dir / f"{dataset_key}_C{cluster:02d}_cell_ids.txt"
    id_table.to_csv(id_csv, index=False)
    id_txt.write_text("\n".join(map(str, ids)) + "\n")
    print("Saved cell ID table:", id_csv)
    print("Saved cell ID text:", id_txt)

    figs = {}
    for view in views:
        print("=" * 80)
        print(f"Rendering fixed 3D {view} view | {dataset_key} C{cluster:02d}")
        print("=" * 80)
        figs[view] = make_fixed_3d_cluster_view(
            dataset_key=dataset_key,
            cluster=cluster,
            view=view,
            render_mode=render_mode,
            show_context=show_context,
            show_soma_text=show_soma_text,
            cell_line_width=cell_line_width,
            context_line_width=context_line_width,
            shell_alpha=shell_alpha,
            max_cells=max_cells,
            out_dir=out_dir,
            save_outputs=save_outputs,
            display_figure=display_figures,
        )

    display(id_table)
    return figs, id_table


def plot_embedding_figure_ready(dataset_key=FIGURE_DATASET_KEY, method="umap", label_clusters=FIGURE_UMAP_LABEL_CLUSTERS, label_points=FIGURE_UMAP_LABEL_POINTS):
    """Clean UMAP/MDS plot with stable cluster colors."""
    if dataset_key not in datasets:
        print(f"Skipping {dataset_key}: dataset not loaded.")
        return None
    ds = datasets[dataset_key]
    if "embeddings" not in ds:
        print(f"Skipping {dataset_key}: no embeddings available.")
        return None
    emb = ds["embeddings"].get(method)
    if emb is None or len(emb) == 0:
        print(f"Skipping {dataset_key} {method}: embedding not available.")
        return None

    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    emb = emb.copy()
    emb["bodyId"] = emb["bodyId"].astype(int)
    df = emb.merge(a[["bodyId", "cluster", "cluster_name", "cluster_size"]], on="bodyId", how="left")
    df = df.dropna(subset=["cluster"])
    df["cluster"] = df["cluster"].astype(int)

    xcol = f"{method}1" if f"{method}1" in df.columns else df.columns[1]
    ycol = f"{method}2" if f"{method}2" in df.columns else df.columns[2]

    fig, ax = plt.subplots(figsize=(5.2, 4.45), dpi=450)
    for cl, sub in df.groupby("cluster", sort=True):
        ax.scatter(
            sub[xcol], sub[ycol],
            s=FIGURE_UMAP_POINT_SIZE,
            alpha=FIGURE_UMAP_ALPHA,
            c=get_cluster_color(dataset_key, int(cl)),
            edgecolors="black",
            linewidths=FIGURE_UMAP_EDGE_LW,
            label=f"C{int(cl):02d} (n={len(sub)})",
        )
        if label_clusters:
            cx, cy = sub[xcol].median(), sub[ycol].median()
            ax.text(cx, cy, f"C{int(cl):02d}", fontsize=7, weight="bold", ha="center", va="center")

    if label_points:
        for _, r in df.iterrows():
            ax.text(r[xcol], r[ycol], str(int(r["bodyId"])), fontsize=4.5, alpha=0.75)

    ax.set_title(f"{ds['label']} — {method.upper()} by morphology cluster")
    ax.set_xlabel(f"{method.upper()} 1")
    ax.set_ylabel(f"{method.upper()} 2")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(False)

    # Keep legend out of the way for many clusters.
    if df["cluster"].nunique() <= 18:
        ax.legend(frameon=False, fontsize=6, loc="center left", bbox_to_anchor=(1.02, 0.5))

    out_base = FIG_UMAP_DIR / f"{dataset_key}_{method}_clusters_figure_ready"
    save_matplotlib_all_formats(fig, out_base, dpi=450)
    plt.show()
    return fig, df


In [ ]:

# ============================================================
# 13. Publication-style UMAP/MDS plots — soma-in-ROI dataset
# ============================================================

soma_in_roi_umap_result = plot_embedding_figure_ready(FIGURE_DATASET_KEY, "umap")
soma_in_roi_mds_result = plot_embedding_figure_ready(FIGURE_DATASET_KEY, "mds")


In [ ]:

# ============================================================
# 14. Batch-render top soma-in-ROI clusters as fixed 3D PDF/PNG/SVG views
# ============================================================

def render_top_fixed_3d_cluster_views(dataset_key=FIGURE_DATASET_KEY):
    """Render the largest clusters for the soma-in-Raphe-superior dataset."""
    if dataset_key not in datasets:
        print(f"{dataset_key} dataset not loaded.")
        return []

    ds = datasets[dataset_key]
    sizes = ds["assignments"]["cluster"].value_counts().sort_values(ascending=False)
    sizes = sizes.loc[sizes >= FIGURE_MIN_CLUSTER_SIZE_TO_RENDER]
    clusters = [int(c) for c in sizes.head(FIGURE_TOP_N_CLUSTERS).index]

    print(f"Rendering top {len(clusters)} fixed 3D cluster panels for {dataset_key}:")
    display(sizes.head(FIGURE_TOP_N_CLUSTERS).to_frame("n_cells"))

    outputs = []
    for cl in clusters:
        out_dir = FIG_CLUSTER_3D_DIR / dataset_key / f"C{cl:02d}"
        figs, tab = make_fixed_3d_cluster_views(
            dataset_key=dataset_key,
            cluster=cl,
            views=FIGURE_3D_VIEWS,
            render_mode=FIGURE_3D_RENDER_MODE,
            show_context=FIGURE_3D_SHOW_CONTEXT,
            show_soma_text=FIGURE_3D_SHOW_SOMA_TEXT,
            cell_line_width=FIGURE_3D_CELL_LINE_WIDTH,
            context_line_width=FIGURE_3D_CONTEXT_LINE_WIDTH,
            shell_alpha=FIGURE_3D_SHELL_ALPHA,
            max_cells=FIGURE_3D_MAX_CELLS,
            out_dir=out_dir,
            save_outputs=True,
            display_figures=False,
        )
        outputs.append(out_dir)
    return outputs

# Run batch export.
soma_in_roi_fixed_3d_outputs = render_top_fixed_3d_cluster_views(FIGURE_DATASET_KEY)


In [ ]:

# ============================================================
# 15. Custom figure-ready fixed 3D plot for one cluster
# ============================================================
# Edit these parameters and rerun this cell.

CUSTOM_DATASET_KEY = FIGURE_DATASET_KEY
CUSTOM_CLUSTER_ID = 1
CUSTOM_3D_VIEWS = ["top", "side", "angled"]
CUSTOM_RENDER_MODE_3D = "combined"  # "skeletons_only", "somas_only", or "combined"
CUSTOM_SHOW_CONTEXT = False
CUSTOM_SHOW_SOMA_TEXT_IN_3D = False
CUSTOM_CELL_LINE_WIDTH = 3.0
CUSTOM_CONTEXT_LINE_WIDTH = 0.45
CUSTOM_SHELL_ALPHA = 0.060
CUSTOM_MAX_CELLS = 120


def make_custom_cluster_fixed_3d_figure_ready_plot(
    dataset_key=CUSTOM_DATASET_KEY,
    cluster=CUSTOM_CLUSTER_ID,
    views=CUSTOM_3D_VIEWS,
    render_mode_3d=CUSTOM_RENDER_MODE_3D,
    show_context=CUSTOM_SHOW_CONTEXT,
    show_soma_text_3d=CUSTOM_SHOW_SOMA_TEXT_IN_3D,
    cell_line_width=CUSTOM_CELL_LINE_WIDTH,
    context_line_width=CUSTOM_CONTEXT_LINE_WIDTH,
    shell_alpha=CUSTOM_SHELL_ALPHA,
    max_cells=CUSTOM_MAX_CELLS,
):
    """
    Create figure-ready fixed-camera 3D PDFs/PNGs/SVGs for one cluster.

    Unlike the old projection panel, this uses the same nice interactive 3D
    outer brain shell, just with fixed cameras for static export.
    """
    if dataset_key not in datasets:
        raise ValueError(f"Unknown dataset_key={dataset_key!r}. Available: {list(datasets.keys())}")

    ds = datasets[dataset_key]
    cluster = int(cluster)
    ids = cluster_body_ids(ds, cluster)
    if len(ids) == 0:
        raise ValueError(f"No cells found for {dataset_key} cluster {cluster}")

    custom_dir = FIG_SINGLE_CLUSTER_DIR / dataset_key / f"C{cluster:02d}"
    custom_dir.mkdir(parents=True, exist_ok=True)

    print(f"{ds['label']} | C{cluster:02d} | n={len(ids)}")
    display(pd.DataFrame({"bodyId": ids}))

    figs, id_table = make_fixed_3d_cluster_views(
        dataset_key=dataset_key,
        cluster=cluster,
        views=views,
        render_mode=render_mode_3d,
        show_context=show_context,
        show_soma_text=show_soma_text_3d,
        cell_line_width=cell_line_width,
        context_line_width=context_line_width,
        shell_alpha=shell_alpha,
        max_cells=max_cells,
        out_dir=custom_dir,
        save_outputs=True,
        display_figures=True,
    )

    return figs, id_table


custom_fixed_3d_figs, custom_fixed_3d_cell_table = make_custom_cluster_fixed_3d_figure_ready_plot()
